# Notebook 11 — StratLake Expanded Promotion Evidence Review

This is the cleaned, source-safe repository import of Notebook 11 for `christophermoverton/fintech-stratlake-notebook-workflows`.

**Theme:** From confidence review to promotion evidence.

Notebook 11 consumes Notebook 10 smoke-mode review artifacts and asks what additional expanded evidence would be required before a strategy could responsibly move from `needs_review` toward human watchlist review or promotion candidacy.

## Non-claims

This notebook does **not** introduce a new promotion engine. It does **not** claim alpha, production readiness, statistical significance, strategy approval, complete artifact coverage, checkpoint generality, CI/runtime equivalence, or promotion-grade evidence by default.

The default posture is conservative: review available artifacts first, preview expanded execution plans second, and frame results as expanded evidence review, caveat/blocker review, and promotion-readiness interpretation. Expanded validation runs only when explicitly enabled.



> **v17 audit posture.** This source import keeps `expanded_run` as a supported successful path, but restores source-safe defaults to `expanded_preview`. Strategy execution, archive restore, governance execution, and checkpointing should be enabled intentionally. Expanded-run success is interpreted separately from complete platform promotion evidence: metrics plus Notebook 11 interpretive packages are useful evidence, but complete platform evidence still requires split metrics and promotion-gate artifacts.

## Audit revision notes

This notebook source preserves the audited raw workflow through v17.

Key v12 corrections:

- Source-safe default is restored to `expanded_preview`.
- `expanded_run` remains a supported live path and can be activated explicitly.
- Successful expanded strategy commands are no longer treated as complete review evidence by themselves.
- Expanded artifact loading now separates:
  - command success,
  - run-id metric artifacts,
  - platform review artifacts,
  - Notebook 11 interpretive review packages.
- When platform split/readiness/gate artifacts are unavailable, Notebook 11 can write conservative notebook-scoped review packages under its own review directory without pretending these are first-class StratLake promotion-engine outputs.
- Governance execution remains optional and schema-discovery-first.
- Archive checkpoint remains off by default.

### v17 audit update — reference-only context gating

This source import keeps source-safe preview mode, but reference-summary fallback rows no longer become expanded-plan candidates by default. Restore Notebook 10 artifacts, run explicit expanded mode, or set `ALLOW_REFERENCE_ONLY_EXPANDED_PLAN=True` when intentionally creating a reference-only plan.


### v14 audit update — run-id strict artifact discovery

This source import keeps `expanded_run` supported, but defaults to `expanded_preview` for source safety. Expanded-run platform artifact discovery is now run-id strict by default so restored Notebook 10 or older strategy-named artifacts are not mistaken for artifacts generated by the current expanded run. Optional governance execution and archive checkpoint remain off unless explicitly enabled.


## 1. Install notebook dependencies and app packages

This install cell is aligned with the working Notebook 10 package pathway. In a fresh Colab runtime, install `pandas-market-calendars`, then install both application packages with the TestPyPI + PyPI fallback pattern. This replacement preserves the substituted package-loading pattern needed for `fintech-market-ingestion` before initializing Fintech and StratLake sessions.

In [ ]:
!pip install -q "pandas-market-calendars>=5.0"
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine


## 2. Imports, Colab detection, and display helpers

These helpers are intentionally source-safe: they load artifacts when present, preview commands before execution, and avoid treating missing optional files as fabricated evidence.

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import getpass
import re
import shlex
from datetime import datetime, timezone
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except Exception:
    drive = None
    userdata = None
    IN_COLAB = False

try:
    import yaml
except Exception:
    yaml = None

print("IN_COLAB:", IN_COLAB)
print("Python executable:", sys.executable)
print("Current working directory:", Path.cwd().as_posix())

if IN_COLAB and drive is not None:
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skipping Google Drive mount.")


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def safe_load_json(path: Path, default: Any = None) -> Any:
    if default is None:
        default = {}
    try:
        if path.exists():
            with path.open("r", encoding="utf-8") as f:
                return json.load(f)
    except Exception as exc:
        print(f"WARN: failed to load JSON {path}: {exc}")
    return default


def safe_load_csv(path: Path, **kwargs) -> pd.DataFrame:
    try:
        if path.exists():
            return pd.read_csv(path, **kwargs)
    except Exception as exc:
        print(f"WARN: failed to load CSV {path}: {exc}")
    return pd.DataFrame()


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=str)


def write_table_pair(df: pd.DataFrame, stem: str, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_dir / f"{stem}.csv", index=False)
    df.to_json(output_dir / f"{stem}.json", orient="records", indent=2)


def preview_command(cmd: list[str]) -> str:
    return " ".join(str(part) for part in cmd)


def run_command(cmd: list[str], cwd: Path | None = None, enabled: bool = False, timeout: int | None = None) -> dict[str, Any]:
    payload = {
        "command": preview_command(cmd),
        "cwd": cwd.as_posix() if cwd else None,
        "enabled": bool(enabled),
        "returncode": None,
        "stdout": "",
        "stderr": "",
        "status": "preview_only",
    }
    print(payload["command"])
    if not enabled:
        return payload
    try:
        result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True, timeout=timeout)
        payload.update({
            "returncode": result.returncode,
            "stdout": result.stdout,
            "stderr": result.stderr,
            "status": "ok" if result.returncode == 0 else "failed",
        })
    except Exception as exc:
        payload.update({"status": "exception", "stderr": repr(exc)})
    return payload


def compact_display(df: pd.DataFrame, rows: int = 20) -> None:
    if df is None or df.empty:
        print("No rows to display.")
    else:
        display(df.head(rows))


## 3. Optional Alpaca environment variables

Notebook 11 primarily reviews restored Notebook 10 evidence. Alpaca credentials are not required for review-only artifact inspection, but this optional guarded cell keeps parity with Notebook 10 for expanded live runs that may validate data access or run upstream feature workflows.

In [ ]:
# Review-only Notebook 11 does not require live Alpaca credentials.
# Set this to True only for expanded live runs that actually validate upstream market-data access.
RUN_LOAD_ALPACA_ENV = False


def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value


if RUN_LOAD_ALPACA_ENV:
    alpaca_api_key_id = get_secret_or_prompt("ALPACA_API_KEY_ID")
    alpaca_api_secret_key = get_secret_or_prompt("ALPACA_API_SECRET_KEY")

    if not alpaca_api_key_id or not alpaca_api_secret_key:
        raise ValueError("Missing Alpaca API credentials.")

    os.environ["ALPACA_API_KEY_ID"] = alpaca_api_key_id
    os.environ["ALPACA_API_SECRET_KEY"] = alpaca_api_secret_key
    os.environ["ALPACA_DATA_BASE_URL"] = "https://data.alpaca.markets"
    os.environ["ALPACA_FEED"] = "iex"

    print("ALPACA_DATA_BASE_URL:", os.environ.get("ALPACA_DATA_BASE_URL"))
    print("ALPACA_FEED:", os.environ.get("ALPACA_FEED"))
    print("ALPACA_API_KEY_ID and ALPACA_API_SECRET_KEY are set but not printed.")
else:
    print("RUN_LOAD_ALPACA_ENV is False; skipping credential prompts for review-only evidence inspection.")


## 4. Configure workspace, sessions, archive paths, and mode

Default mode is `expanded_preview`, which builds an expanded evidence plan without running strategies.

To perform an actual expanded run after reviewing Notebook 10 caveats, set these controls near the top of the notebook, or use matching environment variables before running the notebook:

```python
NOTEBOOK11_MODE = "expanded_run"
RUN_EXPANDED_STRATEGY_EVALUATION = True
ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS = True
```

The expanded-run command shape intentionally mirrors Notebook 10's successful native execution pattern. Archive restore and checkpoint remain manual/off by default.


In [ ]:
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()

# Keep this aligned with Notebook 10. Override with an environment variable when using a different Drive folder.
DRIVE_FOLDER_NAME = os.environ.get("STRATLAKE_DRIVE_FOLDER_NAME", "REPLACE_WITH_DRIVE_FOLDER_NAME")
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME if IN_COLAB else WORKSPACE_ROOT / "drive" / DRIVE_FOLDER_NAME
if IN_COLAB and DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    raise ValueError("Set STRATLAKE_DRIVE_FOLDER_NAME before using Google Drive-backed Notebook 11 runtime paths.")

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"

FINTECH_DRIVE_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
STRATLAKE_DRIVE_ROOT = DRIVE_ROOT / "stratlake-trade-engine"

FINTECH_SESSION_NAME = "fintech_stratlake_input"
STRATLAKE_SESSION_NAME = "stratlake_q1_feature_consumption"

FINTECH_SESSION_ID_OVERRIDE = ""
STRATLAKE_SESSION_ID_OVERRIDE = "stratlake_q1_feature_consumption"

ANALYSIS_START = "2026-01-02"
ANALYSIS_END = "2026-03-31"
BACKFILL_START = "2025-11-03"
BACKFILL_END = "2026-04-15"
FEATURE_BUILD_START = BACKFILL_START
FEATURE_BUILD_END = BACKFILL_END
BACKFILL_SYMBOLS = "AAPL,MSFT,NVDA,SPY,QQQ"

# Mode contract:
# - review_only: inspect existing Notebook 10 artifacts; do not build/run expanded strategy commands.
# - expanded_preview: build expanded evidence plans and artifact expectations; do not execute strategies.
# - expanded_run: execute selected expanded strategy/window commands when RUN_EXPANDED_STRATEGY_EVALUATION=True.
NOTEBOOK11_MODE = os.environ.get("NOTEBOOK11_MODE", "expanded_preview")  # review_only | expanded_preview | expanded_run
if NOTEBOOK11_MODE not in {"review_only", "expanded_preview", "expanded_run"}:
    raise ValueError(f"Unsupported NOTEBOOK11_MODE={NOTEBOOK11_MODE!r}")


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return bool(default)
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}


def env_int_or_none(name: str, default=None):
    value = os.environ.get(name)
    if value is None or str(value).strip() == "":
        return default
    return int(value)

# Notebook 10-style session initialization remains enabled because it creates the
# default configs/session structure that a pure archive-only restore can miss.
RUN_FINTECH_PROJECT_INIT = env_bool("RUN_FINTECH_PROJECT_INIT", True)
RUN_STRATLAKE_SESSION_INIT = env_bool("RUN_STRATLAKE_SESSION_INIT", True)

# Source-safe defaults. Restore is useful for explicit expanded runs, but preview
# mode should not silently restore archives unless the user opts in.
RUN_STRATLAKE_ARCHIVE_RESTORE = env_bool("RUN_STRATLAKE_ARCHIVE_RESTORE", False)
RUN_PROMOTION_GOVERNANCE_REPORT = env_bool("RUN_PROMOTION_GOVERNANCE_REPORT", False)
RUN_STRATLAKE_ARCHIVE_CHECKPOINT = env_bool("RUN_STRATLAKE_ARCHIVE_CHECKPOINT", False)

# Expanded-run controls. In expanded_run mode, the notebook can execute successfully
# once the user explicitly enables strategy execution. Manual-review candidates
# remain gated unless explicitly allowed.
RUN_EXPANDED_STRATEGY_EVALUATION = env_bool(
    "RUN_EXPANDED_STRATEGY_EVALUATION",
    False,
)
ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS = env_bool(
    "ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS",
    False,
)
EXPANDED_RUN_REQUIRE_ALL_ATTEMPTED_SUCCESS = env_bool("EXPANDED_RUN_REQUIRE_ALL_ATTEMPTED_SUCCESS", True)
EXPANDED_RUN_CANDIDATE_LIMIT = env_int_or_none("EXPANDED_RUN_CANDIDATE_LIMIT", None)

# Optional governance command execution is split from schema discovery so help
# checks can be reviewed without accidentally running unstable governance jobs.
RUN_GOVERNANCE_CLI_SCHEMA_DISCOVERY = env_bool("RUN_GOVERNANCE_CLI_SCHEMA_DISCOVERY", True)
RUN_EVIDENCE_REVIEW_CLI_BUILD = env_bool("RUN_EVIDENCE_REVIEW_CLI_BUILD", False)
RUN_PROMOTION_GOVERNANCE_REPORT_CLI = env_bool("RUN_PROMOTION_GOVERNANCE_REPORT_CLI", False)

# Platform artifact discovery should be run-id strict during expanded_run so older
# restored strategy artifacts are not mistaken for artifacts from this execution.
RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY = env_bool("RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY", True)

# Preview mode should not count stale/restored expanded artifacts as if they came
# from this notebook run. Enable this only when intentionally auditing existing
# expanded-run artifacts rather than current-run outputs.
DISCOVER_EXISTING_EXPANDED_PLATFORM_ARTIFACTS = env_bool(
    "DISCOVER_EXISTING_EXPANDED_PLATFORM_ARTIFACTS",
    False,
)

# Notebook-scoped artifact completion package.
# This writes conservative interpretive review packages when expanded-run metrics
# are available but platform split/readiness/gate artifacts are not available.
# These are Notebook 11 review artifacts, not replacements for upstream
# StratLake promotion-engine outputs.
WRITE_NOTEBOOK11_REVIEW_PACKAGES = env_bool("WRITE_NOTEBOOK11_REVIEW_PACKAGES", True)
REQUIRE_PLATFORM_REVIEW_ARTIFACTS_FOR_COMPLETE_PROMOTION_EVIDENCE = env_bool(
    "REQUIRE_PLATFORM_REVIEW_ARTIFACTS_FOR_COMPLETE_PROMOTION_EVIDENCE",
    True,
)

# Notebook 10 reference loading controls.
RUN_NOTEBOOK10_REFERENCE_SUMMARY_FALLBACK = env_bool("RUN_NOTEBOOK10_REFERENCE_SUMMARY_FALLBACK", True)
# Reference-only fallback is useful for source-safe documentation, but it should
# not be treated as sufficient strategy-level evidence for expanded candidate
# selection unless explicitly enabled. Restore Notebook 10 artifacts for the
# normal Notebook 11 candidate screen.
ALLOW_REFERENCE_ONLY_EXPANDED_PLAN = env_bool("ALLOW_REFERENCE_ONLY_EXPANDED_PLAN", False)
AUTO_RESTORE_NOTEBOOK10_CONTEXT_IF_MISSING = env_bool(
    "AUTO_RESTORE_NOTEBOOK10_CONTEXT_IF_MISSING",
    False,
)

# Notebook 11 evidence planning controls.
# Runnable strategies with smoke warnings should usually be surfaced as manual-review
# expanded-plan candidates, even when they are not strict automatic candidates.
EXPANDED_REVIEW_INCLUDE_MANUAL_REVIEW_CANDIDATES = env_bool("EXPANDED_REVIEW_INCLUDE_MANUAL_REVIEW_CANDIDATES", True)
CANDIDATE_STRATEGIES_OVERRIDE = [x.strip() for x in os.environ.get("CANDIDATE_STRATEGIES_OVERRIDE", "").split(",") if x.strip()]
EXPANDED_CANDIDATE_LIMIT = env_int_or_none("EXPANDED_CANDIDATE_LIMIT", EXPANDED_RUN_CANDIDATE_LIMIT)

# Expanded validation windows. Keep one conservative default window aligned with
# Notebook 10's Q1 review interval; add more windows manually only after confirming
# restored feature coverage for the requested date range.
EXPANDED_VALIDATION_WINDOWS = [
    {"window_name": "expanded_q1_review", "start": ANALYSIS_START, "end": ANALYSIS_END},
]

# Defensive source-safe overrides.
if NOTEBOOK11_MODE == "review_only":
    RUN_EXPANDED_STRATEGY_EVALUATION = False
    RUN_PROMOTION_GOVERNANCE_REPORT = False
    RUN_EVIDENCE_REVIEW_CLI_BUILD = False
    RUN_PROMOTION_GOVERNANCE_REPORT_CLI = False
    RUN_STRATLAKE_ARCHIVE_CHECKPOINT = False
    ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS = False
elif NOTEBOOK11_MODE == "expanded_preview":
    RUN_EXPANDED_STRATEGY_EVALUATION = False
    RUN_PROMOTION_GOVERNANCE_REPORT = False
    RUN_EVIDENCE_REVIEW_CLI_BUILD = False
    RUN_PROMOTION_GOVERNANCE_REPORT_CLI = False
    RUN_STRATLAKE_ARCHIVE_CHECKPOINT = False

for path in [FINTECH_ROOT, STRATLAKE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("WORKSPACE_ROOT:", WORKSPACE_ROOT.as_posix())
print("DRIVE_ROOT:", DRIVE_ROOT.as_posix())
print("FINTECH_ROOT:", FINTECH_ROOT.as_posix())
print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("NOTEBOOK11_MODE:", NOTEBOOK11_MODE)
print("Session initialization enabled:", RUN_FINTECH_PROJECT_INIT, RUN_STRATLAKE_SESSION_INIT)
print("Archive restore enabled:", RUN_STRATLAKE_ARCHIVE_RESTORE)
print("Auto restore Notebook 10 context if missing:", AUTO_RESTORE_NOTEBOOK10_CONTEXT_IF_MISSING)
print("Reference-only expanded plan allowed:", ALLOW_REFERENCE_ONLY_EXPANDED_PLAN)
print("Expanded execution enabled:", RUN_EXPANDED_STRATEGY_EVALUATION)
print("Manual-review candidate runs allowed:", ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS)
print("Expanded validation windows:", EXPANDED_VALIDATION_WINDOWS)
print("Governance CLI enabled:", RUN_PROMOTION_GOVERNANCE_REPORT)
print("Governance schema discovery enabled:", RUN_GOVERNANCE_CLI_SCHEMA_DISCOVERY)
print("Notebook 11 review packages enabled:", WRITE_NOTEBOOK11_REVIEW_PACKAGES)
print("Run-id strict platform artifact discovery:", RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY)
print("Existing expanded platform artifact discovery enabled:", DISCOVER_EXISTING_EXPANDED_PLATFORM_ARTIFACTS)
print("Expanded-run recipe: set NOTEBOOK11_MODE=expanded_run, then explicitly enable RUN_EXPANDED_STRATEGY_EVALUATION and ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS after reviewing the plan. Enable archive restore separately only when Notebook 10 artifacts are needed.")
print("Optional governance/checkpoint side effects remain off by default, including in expanded_run, unless their flags are explicitly enabled.")
print("For complete promotion evidence, platform split/readiness/gate artifacts are still required; Notebook 11 packages are interpretive review artifacts only.")


## 5. Verify installed StratLake command and import surfaces

This cell reports surface availability. It does not fail early in `review_only` mode unless you choose to enforce strict runtime requirements.

In [ ]:
CLI_COMMANDS = [
    "stratlake-run-strategy",
    "stratlake-compare-strategies",
    "stratlake-build-evidence-review",
    "stratlake-run-promotion-governance-report",
    "stratlake-session-archive-restore-bootstrap",
    "stratlake-session-archive-bootstrap",
]

cli_rows = []
for cmd in CLI_COMMANDS:
    found = shutil.which(cmd)
    help_status = "not_found"
    returncode = None
    stderr_preview = ""
    if found:
        result = subprocess.run([cmd, "--help"], text=True, capture_output=True)
        returncode = result.returncode
        help_status = "ok" if result.returncode == 0 else "help_failed"
        stderr_preview = (result.stderr or "")[:400]
    cli_rows.append({
        "command": cmd,
        "path": found or "",
        "returncode": returncode,
        "status": help_status,
        "stderr_preview": stderr_preview,
    })

cli_status = pd.DataFrame(cli_rows)
compact_display(cli_status, rows=20)

PYTHON_SURFACES = [
    "src.execution",
    "src.execution.run_strategy",
    "src.execution.compare_strategies",
    "src.execution.run_research_campaign",
    "src.execution.load_json_artifact",
    "src.research.promotion",
]

surface_rows = []
for surface in PYTHON_SURFACES:
    try:
        if "." in surface and surface.count(".") >= 2:
            module_name, attr = surface.rsplit(".", 1)
            module = __import__(module_name, fromlist=[attr])
            ok = hasattr(module, attr)
            surface_rows.append({"surface": surface, "status": "ok" if ok else "missing_attr", "error": ""})
        else:
            __import__(surface)
            surface_rows.append({"surface": surface, "status": "ok", "error": ""})
    except Exception as exc:
        surface_rows.append({"surface": surface, "status": "warn", "error": repr(exc)[:500]})

python_surface_status = pd.DataFrame(surface_rows)
compact_display(python_surface_status, rows=20)

## 6. Initialize or attach Fintech project/session

This mirrors Notebook 10’s initialization pattern and prevents a pure archive restore from leaving default Fintech session directories missing. It creates or attaches the Fintech demo project before StratLake is initialized against the curated MarketLake root.

In [ ]:
fintech_init_cmd = [
    "fintech-init-project",
    "--root", FINTECH_ROOT.as_posix(),
    "--session-name", FINTECH_SESSION_NAME,
    "--with-session",
    "--colab-profile",
]

print("Fintech init command:")
print(" ".join(fintech_init_cmd))

fintech_init_result = run_command(
    fintech_init_cmd,
    cwd=WORKSPACE_ROOT,
    enabled=RUN_FINTECH_PROJECT_INIT,
    timeout=60 * 10,
)

if RUN_FINTECH_PROJECT_INIT and fintech_init_result.get("status") != "ok":
    print("STDOUT:")
    print(fintech_init_result.get("stdout", ""))
    print("STDERR:")
    print(fintech_init_result.get("stderr", ""))
    raise RuntimeError(f"fintech-init-project failed with status {fintech_init_result.get('status')} and return code {fintech_init_result.get('returncode')}")

session_manifest_candidates = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if FINTECH_SESSION_ID_OVERRIDE:
    FINTECH_SESSION_ID = FINTECH_SESSION_ID_OVERRIDE
elif session_manifest_candidates:
    FINTECH_SESSION_ID = session_manifest_candidates[0].parent.name
else:
    FINTECH_SESSION_ID = FINTECH_SESSION_NAME

MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "daily_bars"

FINTECH_ARCHIVE_ID = f"curated-data-{FINTECH_SESSION_ID}"
FINTECH_DRIVE_SESSION_ROOT = FINTECH_DRIVE_ROOT / "sessions" / FINTECH_SESSION_ID
FINTECH_DRIVE_ARCHIVE_ROOT = FINTECH_DRIVE_SESSION_ROOT / "archives"

print("Fintech init status:", fintech_init_result.get("status"))
print("FINTECH_SESSION_ID:", FINTECH_SESSION_ID)
print("MARKETLAKE_ROOT:", MARKETLAKE_ROOT.as_posix())
print("DAILY_BARS_ROOT:", DAILY_BARS_ROOT.as_posix())
print("FINTECH_DRIVE_ARCHIVE_ROOT:", FINTECH_DRIVE_ARCHIVE_ROOT.as_posix())


## 7. Initialize or attach StratLake session

This follows Notebook 10’s `stratlake-init-session` pattern with `--marketlake-root`, Drive persistence, and `--notebook-configs`. The point is to establish default StratLake config/session files before any Notebook 10 archive is restored or reviewed.

In [ ]:
stratlake_init_cmd = [
    "stratlake-init-session",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--project-name", STRATLAKE_SESSION_NAME,
    "--marketlake-root", MARKETLAKE_ROOT.as_posix(),
    "--drive-root", DRIVE_ROOT.as_posix(),
    "--enable-drive-persistence",
    "--notebook-configs",
]

print("StratLake init command:")
print(" ".join(stratlake_init_cmd))

stratlake_init_result = run_command(
    stratlake_init_cmd,
    cwd=WORKSPACE_ROOT,
    enabled=RUN_STRATLAKE_SESSION_INIT,
    timeout=60 * 10,
)

if RUN_STRATLAKE_SESSION_INIT and stratlake_init_result.get("status") != "ok":
    print("STDOUT:")
    print(stratlake_init_result.get("stdout", ""))
    print("STDERR:")
    print(stratlake_init_result.get("stderr", ""))
    raise RuntimeError(f"stratlake-init-session failed with status {stratlake_init_result.get('status')} and return code {stratlake_init_result.get('returncode')}")

STRATLAKE_SESSION_ID = STRATLAKE_SESSION_ID_OVERRIDE or STRATLAKE_SESSION_NAME
STRATLAKE_ARCHIVE_ID = f"stratlake-session-{STRATLAKE_SESSION_ID}"
STRATLAKE_DRIVE_SESSION_ROOT = STRATLAKE_DRIVE_ROOT / "sessions" / STRATLAKE_SESSION_ID
STRATLAKE_DRIVE_ARCHIVE_ROOT = STRATLAKE_DRIVE_SESSION_ROOT / "archives"
UPSTREAM_STRATLAKE_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / STRATLAKE_ARCHIVE_ID

SOURCE_NOTEBOOK10_ARCHIVE_ID = f"notebook-10-walk-forward-promotion-{STRATLAKE_SESSION_ID}"
SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / SOURCE_NOTEBOOK10_ARCHIVE_ID
NOTEBOOK11_ARCHIVE_ID = f"notebook-11-expanded-promotion-evidence-{STRATLAKE_SESSION_ID}"
NOTEBOOK11_ARCHIVE_PACK_DIR = STRATLAKE_DRIVE_ARCHIVE_ROOT / NOTEBOOK11_ARCHIVE_ID

NOTEBOOK10_REVIEW_DIR = STRATLAKE_ROOT / "artifacts" / "notebook_10_walk_forward_promotion_review"
NOTEBOOK11_REVIEW_DIR = STRATLAKE_ROOT / "artifacts" / "notebook_11_expanded_promotion_evidence_review"
EVALUATION_CONFIG = STRATLAKE_ROOT / "configs" / "evaluation.yml"
NOTEBOOK11_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("StratLake init status:", stratlake_init_result.get("status"))
print("STRATLAKE_SESSION_ID:", STRATLAKE_SESSION_ID)
print("NOTEBOOK10_REVIEW_DIR:", NOTEBOOK10_REVIEW_DIR.as_posix())
print("NOTEBOOK11_REVIEW_DIR:", NOTEBOOK11_REVIEW_DIR.as_posix())
print("Upstream StratLake archive pack dir:", UPSTREAM_STRATLAKE_ARCHIVE_PACK_DIR.as_posix())
print("Notebook 10 archive pack dir:", SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.as_posix())


## 8. Optional StratLake archive restore

Restore remains manual/off by default. When enabled, this uses the Notebook 10 archive pack path produced by the prior notebook checkpoint, but only after Fintech and StratLake have been initialized so default configs/session files are present.

In [ ]:
print("STRATLAKE_ROOT:", STRATLAKE_ROOT.as_posix())
print("Notebook 10 archive pack dir:", SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.as_posix())
print("Notebook 10 archive pack exists:", SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.exists())

notebook10_required_artifact_probe = NOTEBOOK10_REVIEW_DIR / "summary.json"
notebook10_context_missing_before_restore = not notebook10_required_artifact_probe.exists()

effective_run_archive_restore = bool(RUN_STRATLAKE_ARCHIVE_RESTORE)
if (
    AUTO_RESTORE_NOTEBOOK10_CONTEXT_IF_MISSING
    and notebook10_context_missing_before_restore
    and SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.exists()
):
    effective_run_archive_restore = True

restore_cmd = [
    "stratlake-session-archive-restore-bootstrap",
    "--archive-root", SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.as_posix(),
    "--target-root", ".",
    "--validate-before-restore",
    "--inspect-before-restore",
    "--overwrite-policy", "overwrite_allowed",
]

restore_caveats = []
if effective_run_archive_restore and not SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.exists():
    restore_caveats.append("notebook10_archive_pack_missing")
    raise FileNotFoundError(
        "Expected Notebook 10 archive pack was not found. "
        "Run the Notebook 10 archive checkpoint first or update STRATLAKE_SESSION_ID_OVERRIDE / DRIVE_FOLDER_NAME. "
        f"Missing: {SOURCE_NOTEBOOK10_ARCHIVE_PACK_DIR.as_posix()}"
    )

restore_result = run_command(
    restore_cmd,
    cwd=STRATLAKE_ROOT,
    enabled=effective_run_archive_restore,
    timeout=60 * 30,
)

if not effective_run_archive_restore:
    restore_caveats.append("archive_restore_not_run_default_off")
elif restore_result.get("status") != "ok":
    restore_caveats.append("archive_restore_failed_or_incomplete")

restore_result_summary = {k: restore_result.get(k) for k in ["command", "enabled", "returncode", "status", "stderr"]}
restore_result_summary["requested_by_flag"] = bool(RUN_STRATLAKE_ARCHIVE_RESTORE)
restore_result_summary["auto_restore_requested"] = bool(AUTO_RESTORE_NOTEBOOK10_CONTEXT_IF_MISSING and notebook10_context_missing_before_restore)
restore_result_summary["notebook10_context_missing_before_restore"] = bool(notebook10_context_missing_before_restore)
restore_result_summary["stderr"] = (restore_result_summary.get("stderr") or "")[:1000]
restore_result_summary


## 9. Verify initialized session files and restored inputs

This check separates missing default/session files from missing Notebook 10 evidence artifacts. In review-only mode, missing restored artifacts are reported later without fabricating evidence; in expanded-run mode, missing core configs or MarketLake roots should be treated as setup blockers.

In [ ]:
initialized_required_paths = [
    STRATLAKE_ROOT / "configs",
    STRATLAKE_ROOT / "configs" / "strategies.yml",
    STRATLAKE_ROOT / "configs" / "evaluation.yml",
    STRATLAKE_ROOT / "artifacts",
    FINTECH_ROOT / "artifacts" / "sessions",
    MARKETLAKE_ROOT,
]

initialized_check_rows = []
for path in initialized_required_paths:
    initialized_check_rows.append({
        "path": path.as_posix(),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else None,
    })

initialized_session_checks = pd.DataFrame(initialized_check_rows)
compact_display(initialized_session_checks, rows=20)

missing_initialized_paths = initialized_session_checks.loc[~initialized_session_checks["exists"], "path"].tolist()
print("Missing initialized/restored setup paths:", missing_initialized_paths)

if NOTEBOOK11_MODE == "expanded_run" and missing_initialized_paths:
    raise FileNotFoundError(f"Missing required initialized/restored paths for expanded_run: {missing_initialized_paths}")


## 10. Locate Notebook 10 handoff and review artifacts

Notebook 11 should not fabricate smoke evidence. Missing Notebook 10 artifacts are reported explicitly and preview mode can continue with caveats.

In [ ]:
EXPECTED_NOTEBOOK10_ARTIFACTS = [
    "walk_forward_results.csv",
    "walk_forward_results.json",
    "robustness_summary.csv",
    "robustness_summary.json",
    "promotion_review.csv",
    "promotion_review.json",
    "preflight_summary.csv",
    "preflight_summary.json",
    "artifact_inventory.csv",
    "artifact_inventory.json",
    "summary.json",
    "smoke_audit_summary.json",
]

notebook10_artifact_rows = []
for name in EXPECTED_NOTEBOOK10_ARTIFACTS:
    path = NOTEBOOK10_REVIEW_DIR / name
    notebook10_artifact_rows.append({
        "artifact": name,
        "path": path.as_posix(),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })

notebook10_artifact_status = pd.DataFrame(notebook10_artifact_rows)
compact_display(notebook10_artifact_status, rows=30)

missing_notebook10_artifacts = notebook10_artifact_status.loc[~notebook10_artifact_status["exists"], "artifact"].tolist()
print("Missing Notebook 10 artifacts:", missing_notebook10_artifacts)

## 11. Load Notebook 10 smoke review summary

The expected known context from the prior milestone included a passing smoke audit with notes, 14 candidates discovered, 11 runnable strategies, 3 skipped strategies, 11 native executions, 11 `needs_review` outcomes, no promoted/watchlist strategies, and 260 artifact inventory rows. This cell shows observed values from runtime artifacts when available.

In [ ]:
NOTEBOOK10_REFERENCE_SUMMARY = {
    "reference_source": "Notebook 10 smoke-mode handoff context",
    "reference_only": True,
    "smoke_audit_status": "pass",
    "candidate_strategy_count_discovered": 14,
    "candidate_strategy_count_selected": 11,
    "preflight_total_count": 14,
    "preflight_runnable_count": 11,
    "preflight_skipped_count": 3,
    "preflight_skipped_strategies": [
        "breakout",
        "residual_momentum",
        "weighted_cross_section_ensemble",
    ],
    "candidate_strategies_selected": [
        "buy_and_hold_v1",
        "cross_section_momentum",
        "mean_reversion",
        "mean_reversion_v1",
        "mean_reversion_v1_safe_2026_q1",
        "momentum_v1",
        "pairs_trading",
        "seeded_random_v1",
        "sma_crossover_v1",
        "time_series_momentum",
        "volatility_regime_momentum",
    ],
    "promotion_decision_counts": {"needs_review": 11},
    "metric_source_counts": {"artifact_json": 11},
    "native_execution_rows": 11,
    "artifact_inventory_rows": 260,
    "diagnostic_counts": {
        "is_flat_or_inactive": 7,
        "benchmark_avoidance_outperformance": 7,
    },
    "warning_category_counts": {
        "benchmark_degenerate_warning": 11,
        "qa_warn": 11,
        "strategy_degenerate_warning": 2,
        "signal_pct_consistency": 1,
        "flat_series_correlation_warning": 7,
    },
    "interpretation": "Reference summary only. Restore Notebook 10 artifacts for runtime evidence.",
}

notebook10_summary = safe_load_json(NOTEBOOK10_REVIEW_DIR / "summary.json", default={})
smoke_audit_summary = safe_load_json(NOTEBOOK10_REVIEW_DIR / "smoke_audit_summary.json", default={})

promotion_review = safe_load_csv(NOTEBOOK10_REVIEW_DIR / "promotion_review.csv")
robustness_summary = safe_load_csv(NOTEBOOK10_REVIEW_DIR / "robustness_summary.csv")
preflight_summary = safe_load_csv(NOTEBOOK10_REVIEW_DIR / "preflight_summary.csv")
walk_forward_results = safe_load_csv(NOTEBOOK10_REVIEW_DIR / "walk_forward_results.csv")
artifact_inventory = safe_load_csv(NOTEBOOK10_REVIEW_DIR / "artifact_inventory.csv")

notebook10_context_source = "restored_artifacts"
runtime_context_loaded = bool(notebook10_summary or smoke_audit_summary or not promotion_review.empty or not walk_forward_results.empty)

if not runtime_context_loaded and RUN_NOTEBOOK10_REFERENCE_SUMMARY_FALLBACK:
    notebook10_context_source = "reference_summary_fallback"
    notebook10_summary = NOTEBOOK10_REFERENCE_SUMMARY.copy()
    smoke_audit_summary = {
        "notebook10_mode": "smoke",
        "smoke_audit_status": NOTEBOOK10_REFERENCE_SUMMARY["smoke_audit_status"],
        "candidate_strategy_count_discovered": NOTEBOOK10_REFERENCE_SUMMARY["candidate_strategy_count_discovered"],
        "preflight_runnable_count": NOTEBOOK10_REFERENCE_SUMMARY["preflight_runnable_count"],
        "preflight_skipped_count": NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_count"],
        "preflight_skipped_strategies": NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_strategies"],
        "native_execution_rows": NOTEBOOK10_REFERENCE_SUMMARY["native_execution_rows"],
        "promotion_decision_counts": NOTEBOOK10_REFERENCE_SUMMARY["promotion_decision_counts"],
        "metric_source_counts": NOTEBOOK10_REFERENCE_SUMMARY["metric_source_counts"],
        "warning_category_counts": NOTEBOOK10_REFERENCE_SUMMARY["warning_category_counts"],
        "diagnostic_counts": NOTEBOOK10_REFERENCE_SUMMARY["diagnostic_counts"],
        "interpretation": NOTEBOOK10_REFERENCE_SUMMARY["interpretation"],
        "reference_only": True,
    }

observed_notebook10_context = {
    "context_source": notebook10_context_source,
    "runtime_context_loaded": runtime_context_loaded,
    "summary_loaded": bool(notebook10_summary),
    "smoke_audit_loaded": bool(smoke_audit_summary),
    "smoke_audit_status": smoke_audit_summary.get("smoke_audit_status") or notebook10_summary.get("smoke_audit_status"),
    "candidate_strategy_count_discovered": notebook10_summary.get("candidate_strategy_count_discovered"),
    "candidate_strategy_count_selected": notebook10_summary.get("candidate_strategy_count_selected") or len(notebook10_summary.get("candidate_strategies_selected", [])),
    "preflight_total_count": notebook10_summary.get("preflight_total_count", len(preflight_summary)),
    "preflight_runnable_count": notebook10_summary.get("preflight_runnable_count") or smoke_audit_summary.get("preflight_runnable_count"),
    "preflight_skipped_count": notebook10_summary.get("preflight_skipped_count") or smoke_audit_summary.get("preflight_skipped_count"),
    "walk_forward_rows": len(walk_forward_results),
    "promotion_review_rows": len(promotion_review),
    "promotion_decision_counts": (
        promotion_review["promotion_decision"].value_counts(dropna=False).to_dict()
        if "promotion_decision" in promotion_review.columns
        else notebook10_summary.get("promotion_decision_counts", {})
    ),
    "artifact_inventory_rows": len(artifact_inventory),
    "reference_only": notebook10_context_source == "reference_summary_fallback",
}

reference_checks = [
    ("smoke_audit_status", NOTEBOOK10_REFERENCE_SUMMARY["smoke_audit_status"], observed_notebook10_context.get("smoke_audit_status")),
    ("candidate_strategy_count_discovered", NOTEBOOK10_REFERENCE_SUMMARY["candidate_strategy_count_discovered"], observed_notebook10_context.get("candidate_strategy_count_discovered")),
    ("candidate_strategy_count_selected", NOTEBOOK10_REFERENCE_SUMMARY["candidate_strategy_count_selected"], observed_notebook10_context.get("candidate_strategy_count_selected")),
    ("preflight_runnable_count", NOTEBOOK10_REFERENCE_SUMMARY["preflight_runnable_count"], observed_notebook10_context.get("preflight_runnable_count")),
    ("preflight_skipped_count", NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_count"], observed_notebook10_context.get("preflight_skipped_count")),
    ("promotion_review_rows", NOTEBOOK10_REFERENCE_SUMMARY["candidate_strategy_count_selected"], observed_notebook10_context.get("promotion_review_rows")),
    ("artifact_inventory_rows", NOTEBOOK10_REFERENCE_SUMMARY["artifact_inventory_rows"], observed_notebook10_context.get("artifact_inventory_rows")),
]
notebook10_reference_audit = pd.DataFrame([
    {
        "check": name,
        "expected_notebook10_reference": expected,
        "observed_loaded_context": observed,
        "status": "pass" if str(expected) == str(observed) else ("not_observed" if observed in {None, ""} else "review"),
    }
    for name, expected, observed in reference_checks
])

print(json.dumps(observed_notebook10_context, indent=2, default=str))
print("Notebook 10 reference-context audit:")
compact_display(notebook10_reference_audit, rows=20)

if not promotion_review.empty:
    print("Promotion review sample:")
    compact_display(promotion_review, rows=20)
elif notebook10_context_source == "reference_summary_fallback":
    print("Using reference summary fallback. Restore Notebook 10 artifacts for strategy-level evidence.")


## 12. Reconstruct Notebook 10 candidate evidence table

This table translates Notebook 10 smoke outputs into candidate-level evidence for Notebook 11. It is a decision-support table, not a promotion decision.

In [ ]:
def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def as_bool(value: Any) -> bool:
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y", "ok", "passed", "pass"}


def split_reason_codes(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, list):
        return [str(v) for v in value if str(v)]
    text = str(value).strip()
    if not text:
        return []
    for sep in ["|", ";", ","]:
        if sep in text:
            return [part.strip() for part in text.split(sep) if part.strip()]
    return [text]


def warning_categories_from_row(row: pd.Series) -> list[str]:
    cats = []
    for col in ["warning_categories", "reason_codes", "promotion_reasons", "failure_reasons", "preflight_note"]:
        if col in row.index:
            cats.extend(split_reason_codes(row.get(col)))
    return sorted(set(cats))


def value_from_first(row_or_df: Any, candidates: list[str], default: Any = "") -> Any:
    if isinstance(row_or_df, pd.Series):
        for col in candidates:
            if col in row_or_df.index:
                value = row_or_df.get(col)
                if value is not None and not (isinstance(value, float) and pd.isna(value)):
                    return value
    elif isinstance(row_or_df, pd.DataFrame) and not row_or_df.empty:
        for col in candidates:
            if col in row_or_df.columns:
                values = row_or_df[col].dropna()
                if len(values) > 0:
                    return values.iloc[0]
    return default


strategy_col_candidates = ["strategy_name", "strategy", "name"]
preflight_strategy_col = first_existing_column(preflight_summary, strategy_col_candidates)
promotion_strategy_col = first_existing_column(promotion_review, strategy_col_candidates)
robustness_strategy_col = first_existing_column(robustness_summary, strategy_col_candidates)
walk_strategy_col = first_existing_column(walk_forward_results, strategy_col_candidates)

candidate_names = set()
for df, col in [
    (preflight_summary, preflight_strategy_col),
    (promotion_review, promotion_strategy_col),
    (robustness_summary, robustness_strategy_col),
    (walk_forward_results, walk_strategy_col),
]:
    if df is not None and not df.empty and col:
        candidate_names.update(df[col].dropna().astype(str).tolist())

if not candidate_names and observed_notebook10_context.get("reference_only"):
    candidate_names.update(NOTEBOOK10_REFERENCE_SUMMARY["candidate_strategies_selected"])
    candidate_names.update(NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_strategies"])

candidate_rows = []
for strategy in sorted(candidate_names):
    pre_row = pd.Series(dtype=object)
    promo_row = pd.Series(dtype=object)
    robust_row = pd.Series(dtype=object)
    walk_rows = pd.DataFrame()
    row_source = "restored_artifacts"

    if preflight_strategy_col and not preflight_summary.empty:
        matches = preflight_summary[preflight_summary[preflight_strategy_col].astype(str) == strategy]
        if not matches.empty:
            pre_row = matches.iloc[0]
    if promotion_strategy_col and not promotion_review.empty:
        matches = promotion_review[promotion_review[promotion_strategy_col].astype(str) == strategy]
        if not matches.empty:
            promo_row = matches.iloc[0]
    if robustness_strategy_col and not robustness_summary.empty:
        matches = robustness_summary[robustness_summary[robustness_strategy_col].astype(str) == strategy]
        if not matches.empty:
            robust_row = matches.iloc[0]
    if walk_strategy_col and not walk_forward_results.empty:
        walk_rows = walk_forward_results[walk_forward_results[walk_strategy_col].astype(str) == strategy]

    missing_requirements = value_from_first(pre_row, ["missing_requirements", "missing_required_columns", "missing_columns"], "")
    preflight_runnable_value = value_from_first(pre_row, ["preflight_runnable", "runnable", "is_runnable"], None)
    preflight_note = value_from_first(pre_row, ["preflight_note", "status", "preflight_status"], "")

    if preflight_runnable_value is not None and not (isinstance(preflight_runnable_value, float) and pd.isna(preflight_runnable_value)):
        preflight_status = "runnable" if as_bool(preflight_runnable_value) else "skipped_missing_requirements"
    else:
        preflight_status = value_from_first(pre_row, ["preflight_status", "status"], "unknown")

    native_status = "not_observed"
    if not walk_rows.empty:
        rc_col = first_existing_column(walk_rows, ["returncode", "native_returncode"])
        completed_col = first_existing_column(walk_rows, ["completed", "native_execution_completed"])
        if rc_col and pd.to_numeric(walk_rows[rc_col], errors="coerce").fillna(-1).eq(0).all():
            native_status = "completed_returncode_0"
        elif completed_col and walk_rows[completed_col].map(as_bool).all():
            native_status = "completed"
        else:
            native_status = "warning_or_failed"

    metric_source = (
        value_from_first(walk_rows, ["metric_source"], "")
        or value_from_first(promo_row, ["metric_source"], "")
        or value_from_first(robust_row, ["metric_source"], "")
    )

    warning_categories = set(warning_categories_from_row(pre_row) + warning_categories_from_row(promo_row) + warning_categories_from_row(robust_row))
    if not walk_rows.empty:
        for _, wr in walk_rows.iterrows():
            warning_categories.update(warning_categories_from_row(wr))

    flat_col = first_existing_column(walk_rows, ["is_flat_or_inactive"]) if not walk_rows.empty else None
    bench_avoid_col = first_existing_column(walk_rows, ["benchmark_avoidance_outperformance"]) if not walk_rows.empty else None
    active_neg_col = first_existing_column(walk_rows, ["active_negative_return"]) if not walk_rows.empty else None
    qa_clean_col = first_existing_column(walk_rows, ["qa_is_clean", "qa_clean", "is_qa_clean"]) if not walk_rows.empty else None

    if observed_notebook10_context.get("reference_only"):
        row_source = "reference_summary_fallback"
        skipped = strategy in NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_strategies"]
        preflight_status = "skipped_missing_requirements" if skipped else "runnable_reference_only"
        missing_requirements = "see_notebook10_preflight_artifacts" if skipped else ""
        native_status = "not_run_preflight_skipped" if skipped else "completed_reference_only"
        metric_source = "" if skipped else "artifact_json_reference_only"

    candidate_rows.append({
        "strategy_name": strategy,
        "row_source": row_source,
        "preflight_status": preflight_status,
        "preflight_note": preflight_note,
        "missing_requirements": missing_requirements,
        "native_execution_status": native_status,
        "metric_source": metric_source,
        "promotion_decision": value_from_first(promo_row, ["promotion_decision", "decision"], "needs_review" if observed_notebook10_context.get("reference_only") and strategy not in NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_strategies"] else "unknown"),
        "warning_categories": "|".join(sorted(warning_categories)),
        "is_flat_or_inactive": bool(walk_rows[flat_col].map(as_bool).any()) if flat_col else False,
        "benchmark_avoidance_outperformance": bool(walk_rows[bench_avoid_col].map(as_bool).any()) if bench_avoid_col else False,
        "active_negative_return": bool(walk_rows[active_neg_col].map(as_bool).any()) if active_neg_col else False,
        "qa_is_clean": bool(walk_rows[qa_clean_col].map(as_bool).any()) if qa_clean_col else False,
        "smoke_window_count": int(len(walk_rows)) if not observed_notebook10_context.get("reference_only") else (0 if strategy in NOTEBOOK10_REFERENCE_SUMMARY["preflight_skipped_strategies"] else 1),
    })

candidate_evidence_table = pd.DataFrame(candidate_rows)
compact_display(candidate_evidence_table, rows=50)


## 13. Define expanded evidence criteria

These criteria select candidates for deeper review only. They do not promote strategies.

In [ ]:
def classify_expanded_candidacy(row: pd.Series) -> tuple[str, str]:
    reasons = []
    status = "expanded_candidate"

    row_source = str(row.get("row_source", "")).lower()
    preflight_status = str(row.get("preflight_status", "")).lower()
    native_status = str(row.get("native_execution_status", "")).lower()
    missing_requirements = str(row.get("missing_requirements", "")).strip()
    metric_source = str(row.get("metric_source", "")).strip()
    warnings = set(split_reason_codes(row.get("warning_categories")))

    if row_source == "reference_summary_fallback":
        if "skipped" in preflight_status or missing_requirements:
            return "needs_feature_contract_work", "reference_only_missing_feature_context"
        if not ALLOW_REFERENCE_ONLY_EXPANDED_PLAN:
            return "deferred_reference_only", "restore_notebook10_artifacts_before_strategy_level_expanded_plan"
        return "needs_manual_review", "reference_summary_only_restore_notebook10_artifacts"

    if missing_requirements and missing_requirements.lower() not in {"nan", "none", "[]"}:
        return "needs_feature_contract_work", "missing_required_features_or_columns"
    if "fail" in preflight_status or "skip" in preflight_status or "blocked" in preflight_status:
        return "blocked_runtime_or_missing_features", "preflight_not_runnable"
    if native_status in {"warning_or_failed", "not_observed"}:
        reasons.append("native_execution_not_cleanly_observed")
    if bool(row.get("is_flat_or_inactive")):
        return "blocked_flat_or_inactive", "flat_or_inactive_smoke_behavior"
    if bool(row.get("benchmark_avoidance_outperformance")):
        return "blocked_benchmark_avoidance_only", "outperformance_from_benchmark_avoidance"
    if not metric_source or metric_source.lower() in {"nan", "none", "unknown"}:
        reasons.append("missing_or_unknown_metric_source")
    if bool(row.get("active_negative_return")):
        reasons.append("active_negative_return_observed")
    if warnings:
        reasons.append("smoke_warning_categories_present")
    if int(row.get("smoke_window_count") or 0) <= 0:
        reasons.append("no_smoke_window_rows")

    if reasons:
        status = "needs_manual_review"
    return status, "|".join(sorted(set(reasons))) if reasons else "eligible_for_expanded_evidence_preview"


def expanded_plan_candidate_from_label(label: str) -> bool:
    if label == "expanded_candidate":
        return True
    if EXPANDED_REVIEW_INCLUDE_MANUAL_REVIEW_CANDIDATES and label == "needs_manual_review":
        return True
    # Reference-only rows are intentionally excluded from expanded-plan preview
    # unless ALLOW_REFERENCE_ONLY_EXPANDED_PLAN is set before classification.
    return False


if candidate_evidence_table.empty:
    candidate_evidence_table["candidate_for_expanded_review"] = pd.Series(dtype=str)
    candidate_evidence_table["expanded_review_reason"] = pd.Series(dtype=str)
    candidate_evidence_table["expanded_plan_candidate"] = pd.Series(dtype=bool)
    candidate_evidence_table["expanded_plan_gate"] = pd.Series(dtype=str)
else:
    labels = candidate_evidence_table.apply(classify_expanded_candidacy, axis=1, result_type="expand")
    candidate_evidence_table["candidate_for_expanded_review"] = labels[0]
    candidate_evidence_table["expanded_review_reason"] = labels[1]
    candidate_evidence_table["expanded_plan_candidate"] = candidate_evidence_table["candidate_for_expanded_review"].map(expanded_plan_candidate_from_label)
    candidate_evidence_table["expanded_plan_gate"] = np.where(
        candidate_evidence_table["candidate_for_expanded_review"].eq("expanded_candidate"),
        "eligible_for_preview",
        np.where(
            candidate_evidence_table["expanded_plan_candidate"],
            "manual_review_required_before_run",
            "blocked_or_deferred",
        ),
    )

expanded_candidate_names = candidate_evidence_table.loc[
    candidate_evidence_table.get("expanded_plan_candidate", pd.Series(dtype=bool)).eq(True),
    "strategy_name",
].astype(str).tolist() if not candidate_evidence_table.empty else []

strict_expanded_candidate_names = candidate_evidence_table.loc[
    candidate_evidence_table.get("candidate_for_expanded_review", pd.Series(dtype=str)).eq("expanded_candidate"),
    "strategy_name",
].astype(str).tolist() if not candidate_evidence_table.empty else []

if CANDIDATE_STRATEGIES_OVERRIDE:
    expanded_candidate_names = CANDIDATE_STRATEGIES_OVERRIDE
if EXPANDED_CANDIDATE_LIMIT is not None:
    expanded_candidate_names = expanded_candidate_names[: int(EXPANDED_CANDIDATE_LIMIT)]

print("Strict expanded candidate count:", len(strict_expanded_candidate_names))
print("Expanded plan candidate count:", len(expanded_candidate_names))
print(expanded_candidate_names)
compact_display(candidate_evidence_table, rows=50)


## 14. Preview expanded walk-forward execution plan

This section composes existing StratLake strategy execution surfaces. The command shape mirrors Notebook 10's successful native execution pattern rather than inventing a new notebook-only command style:

```bash
stratlake-run-strategy \
  --strategies-config configs/strategies.yml \
  --strategy <strategy_name> \
  --start <window_start> \
  --end <window_end>
```

The plan contains one row per strategy/window. In `expanded_preview`, rows are preview-only. In `expanded_run`, rows execute only when `RUN_EXPANDED_STRATEGY_EVALUATION=True`; manual-review candidates additionally require `ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS=True`.


In [ ]:
execution_plan_rows = []
candidate_lookup = candidate_evidence_table.set_index("strategy_name").to_dict(orient="index") if not candidate_evidence_table.empty else {}

STRATEGIES_CONFIG = STRATLAKE_ROOT / "configs" / "strategies.yml"

for strategy in expanded_candidate_names:
    candidate_meta = candidate_lookup.get(strategy, {})
    candidate_label = candidate_meta.get("candidate_for_expanded_review", "unknown")
    requires_manual_review = candidate_label == "needs_manual_review"
    artifact_root = STRATLAKE_ROOT / "artifacts" / "strategies" / strategy

    for window in EXPANDED_VALIDATION_WINDOWS:
        window_name = str(window.get("window_name", "expanded_window"))
        window_start = str(window.get("start", ANALYSIS_START))
        window_end = str(window.get("end", ANALYSIS_END))
        cmd = [
            "stratlake-run-strategy",
            "--strategies-config", "configs/strategies.yml",
            "--strategy", strategy,
            "--start", window_start,
            "--end", window_end,
        ]
        execution_plan_rows.append({
            "strategy_name": strategy,
            "window_name": window_name,
            "window_start": window_start,
            "window_end": window_end,
            "candidate_label": candidate_label,
            "expanded_review_reason": candidate_meta.get("expanded_review_reason", ""),
            "requires_manual_review_before_run": bool(requires_manual_review),
            "execution_surface": "cli:stratlake-run-strategy",
            "command_style": "notebook10_native_cli",
            "command": preview_command(cmd),
            "command_args": cmd,
            "python_api_preview": f'from src.execution import run_strategy; result = run_strategy("{strategy}", start="{window_start}", end="{window_end}")',
            "required_config": STRATEGIES_CONFIG.as_posix(),
            "required_config_exists": STRATEGIES_CONFIG.exists(),
            "expected_artifact_root": artifact_root.as_posix(),
            "expected_metrics_json": (artifact_root / "metrics.json").as_posix(),
            "expected_metrics_by_split_csv": (artifact_root / "metrics_by_split.csv").as_posix(),
            "expected_metrics_readiness_json": (artifact_root / "metrics_readiness.json").as_posix(),
            "expected_promotion_gates_json": (artifact_root / "promotion_gates.json").as_posix(),
            "run_enabled": bool(RUN_EXPANDED_STRATEGY_EVALUATION),
            "run_blocked_by_manual_review_gate": bool(requires_manual_review and not ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS),
        })

expanded_execution_plan = pd.DataFrame(execution_plan_rows)
compact_display(expanded_execution_plan, rows=100)


## 15. Optional expanded walk-forward execution

This guarded cell executes selected expanded validation rows. A successful expanded run requires attempted rows with `returncode == 0`; skipped/manual-review rows are not counted as successful execution.

Recommended successful-run preset after reviewing Notebook 10 caveats:

```python
NOTEBOOK11_MODE = "expanded_run"
RUN_EXPANDED_STRATEGY_EVALUATION = True
ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS = True
```

For a short smoke of the expanded-run path, set `EXPANDED_RUN_CANDIDATE_LIMIT=1` or `EXPANDED_CANDIDATE_LIMIT=1` before running the notebook.


In [ ]:
# Expanded execution result parsing mirrors Notebook 10's native strategy-output parser.
# Return-code success is necessary, but not sufficient for evidence sufficiency: we also
# parse stdout metrics and search run-id-linked artifacts where available.

def extract(pattern: str, text: str, default=None, cast=None):
    match = re.search(pattern, text or "", flags=re.MULTILINE)
    if not match:
        return default
    value = match.group(1).strip()
    if cast is None:
        return value
    try:
        return cast(value)
    except Exception:
        return default


def extract_percent(pattern: str, text: str, default=None):
    value = extract(pattern, text, default=default, cast=float)
    if value is None:
        return default
    return value / 100.0


def flatten_dict(obj: Any, prefix: str = "") -> dict[str, Any]:
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_dict(v, key))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            key = f"{prefix}.{i}" if prefix else str(i)
            out.update(flatten_dict(v, key))
    else:
        out[prefix] = obj
    return out


def coerce_float(value: Any):
    if value is None or value == "":
        return None
    try:
        return float(value)
    except Exception:
        return None


METRIC_ALIASES = {
    "cumulative_return": ["cumulative_return", "total_return", "strategy_return"],
    "sharpe_ratio": ["sharpe_ratio", "sharpe"],
    "benchmark_return": ["benchmark_return"],
    "excess_return": ["excess_return", "active_return"],
    "turnover": ["turnover"],
    "trades": ["trades", "trade_count", "n_trades"],
    "correlation": ["correlation", "benchmark_correlation"],
}


def find_metric_in_flat(flat: dict[str, Any], aliases: list[str]):
    lower_items = {str(k).lower(): v for k, v in flat.items()}
    for alias in aliases:
        alias_l = alias.lower()
        for k, v in lower_items.items():
            if k == alias_l or k.endswith("." + alias_l):
                return v
    return None


def load_artifact_metrics_for_run(run_id: str | None) -> tuple[dict[str, Any], str]:
    """Search run-id-linked artifacts, matching Notebook 10's artifact-first pattern."""
    if not run_id:
        return {}, "stdout"

    search_roots = [STRATLAKE_ROOT / "artifacts", STRATLAKE_ROOT / "reports", STRATLAKE_ROOT / "data"]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and str(run_id) in p.as_posix() and p.suffix.lower() in [".json", ".csv"]:
                candidates.append(p)

    for p in sorted(candidates, key=lambda x: (x.suffix.lower() != ".json", len(x.as_posix()))):
        try:
            if p.suffix.lower() == ".json":
                payload = json.loads(p.read_text(encoding="utf-8"))
                flat = flatten_dict(payload)
                metrics = {}
                for canonical, aliases in METRIC_ALIASES.items():
                    v = find_metric_in_flat(flat, aliases)
                    if v is not None:
                        metrics[canonical] = coerce_float(v) if canonical != "trades" else int(float(v))
                if metrics:
                    metrics["artifact_metric_path"] = p.as_posix()
                    metrics["artifact_payload"] = payload
                    return metrics, "artifact_json"
            elif p.suffix.lower() == ".csv":
                df = pd.read_csv(p)
                if len(df) > 0:
                    row = df.iloc[-1].to_dict()
                    flat = {str(k): v for k, v in row.items()}
                    metrics = {}
                    for canonical, aliases in METRIC_ALIASES.items():
                        v = find_metric_in_flat(flat, aliases)
                        if v is not None:
                            metrics[canonical] = coerce_float(v) if canonical != "trades" else int(float(v))
                    if metrics:
                        metrics["artifact_metric_path"] = p.as_posix()
                        return metrics, "artifact_csv"
        except Exception:
            continue

    return {}, "stdout"


def parse_expanded_strategy_stdout(strategy_name: str, window_name: str, window_start: str, window_end: str, stdout: str, stderr: str, returncode: int) -> dict[str, Any]:
    parsed = {
        "strategy": extract(r"^strategy:\s*(.+)$", stdout) or strategy_name,
        "window_name": window_name,
        "analysis_start": window_start,
        "analysis_end": window_end,
        "run_id": extract(r"^run_id:\s*(.+)$", stdout),
        "completed": returncode == 0,
        "returncode": returncode,
        "cumulative_return": extract(r"^cumulative_return:\s*([-+0-9.]+)", stdout, cast=float),
        "sharpe_ratio": extract(r"^sharpe_ratio:\s*([-+0-9.]+)", stdout, cast=float),
        "long_pct": extract_percent(r"- long:\s*([-+0-9.]+)%", stdout),
        "short_pct": extract_percent(r"short:\s*([-+0-9.]+)%", stdout),
        "flat_pct": extract_percent(r"flat:\s*([-+0-9.]+)%", stdout),
        "trades": extract(r"- trades:\s*([0-9]+)", stdout, cast=int),
        "turnover": extract(r"turnover:\s*([-+0-9.]+)", stdout, cast=float),
        "avg_holding_bars": extract(r"- avg holding:\s*([-+0-9.]+)\s*bars", stdout, cast=float),
        "qa_status": extract(r"- status:\s*(.+)$", stdout),
        "qa_rows": extract(r"- rows:\s*([0-9]+)", stdout, cast=int),
        "qa_symbols": extract(r"symbols:\s*([0-9]+)", stdout, cast=int),
        "benchmark_return": extract_percent(r"- benchmark return:\s*([-+0-9.]+)%", stdout),
        "excess_return": extract_percent(r"- excess return:\s*([-+0-9.]+)%", stdout),
        "correlation": extract(r"- correlation:\s*([-+0-9.]+)", stdout, cast=float),
    }
    return parsed



def classify_expanded_stderr(stderr: str, stdout: str, returncode: int) -> tuple[str, str]:
    """Conservative warning taxonomy adapted from Notebook 10."""
    raw_text = "\n".join([stderr or "", stdout or ""])
    text = raw_text.lower()
    categories: list[str] = []

    def has_any(*patterns: str) -> bool:
        return any(pattern in text for pattern in patterns)

    if returncode != 0:
        categories.append("runtime_failed")
    if has_any("missing required column", "missing columns", "missing column") or ("missing" in text and "column" in text):
        categories.append("missing_required_columns")
    if "pct_long + pct_short + pct_flat" in text or ("pct" in text and "sum" in text):
        categories.append("signal_pct_consistency")
    if "qa summary" in text and "warn" in text:
        categories.append("qa_warn")
    elif "qa" in text and "warn" in text:
        categories.append("qa_warn")
    if has_any("benchmark buy-and-hold", "buy-and-hold", "buyandholdstrategy", "benchmark strategy") and has_any("always long", "no trades", "degenerate"):
        categories.append("benchmark_degenerate_warning")
    if has_any("strategy appears flat", "flat strategy", "no strategy trades", "zero trades", "always flat", "no trades were generated"):
        categories.append("strategy_degenerate_warning")
    if has_any("constant input", "flat return", "flat series", "invalid value encountered in divide", "invalid value encountered in scalar divide"):
        categories.append("flat_series_correlation_warning")
    numeric_terms = ["runtimewarning", "invalid value", "divide by zero", "overflow encountered", "underflow encountered"]
    if has_any(*numeric_terms) and not any(c in categories for c in ["benchmark_degenerate_warning", "strategy_degenerate_warning", "flat_series_correlation_warning"]):
        categories.append("numeric_runtime_warning")
    if has_any("traceback", "exception") or ("error" in text and returncode != 0):
        categories.append("exception_or_error_text")
    if stderr and not categories:
        categories.append("stderr_other")

    categories = sorted(dict.fromkeys(categories))
    severity = "ok"
    error_categories = {"runtime_failed", "exception_or_error_text", "missing_required_columns"}
    if any(category in error_categories for category in categories):
        severity = "error"
    elif categories:
        severity = "warn"
    return severity, ",".join(categories)


expanded_execution_results = []
for row in execution_plan_rows:
    cmd = row.get("command_args") if isinstance(row.get("command_args"), list) else shlex.split(row["command"])
    execution_key = f"{row.get('strategy_name')}::{row.get('window_name')}"

    if not RUN_EXPANDED_STRATEGY_EVALUATION:
        expanded_execution_results.append({
            "execution_key": execution_key,
            "strategy_name": row["strategy_name"],
            "window_name": row.get("window_name"),
            "window_start": row.get("window_start"),
            "window_end": row.get("window_end"),
            "command": row["command"],
            "enabled": False,
            "attempted": False,
            "returncode": None,
            "status": "preview_only",
            "completed": False,
            "run_id": "",
            "metric_source": "not_run",
            "artifact_metric_path": "",
            "stdout_preview": "",
            "stderr_preview": "Expanded execution is disabled. Review the plan before setting RUN_EXPANDED_STRATEGY_EVALUATION=True.",
        })
        continue

    if row.get("requires_manual_review_before_run") and not ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS:
        expanded_execution_results.append({
            "execution_key": execution_key,
            "strategy_name": row["strategy_name"],
            "window_name": row.get("window_name"),
            "window_start": row.get("window_start"),
            "window_end": row.get("window_end"),
            "command": row["command"],
            "enabled": False,
            "attempted": False,
            "returncode": None,
            "status": "skipped_manual_review_required",
            "completed": False,
            "run_id": "",
            "metric_source": "not_run",
            "artifact_metric_path": "",
            "stdout_preview": "",
            "stderr_preview": "Manual-review candidate. Set ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS=True and RUN_EXPANDED_STRATEGY_EVALUATION=True only after reviewing Notebook 10 caveats.",
        })
        continue

    result = run_command(
        cmd,
        cwd=STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT,
        enabled=True,
        timeout=60 * 30,
    )
    stdout = result.get("stdout") or ""
    stderr = result.get("stderr") or ""
    returncode = result.get("returncode")
    parsed = parse_expanded_strategy_stdout(
        row["strategy_name"],
        row.get("window_name"),
        row.get("window_start"),
        row.get("window_end"),
        stdout,
        stderr,
        int(returncode) if returncode is not None else -1,
    )
    artifact_metrics, metric_source = load_artifact_metrics_for_run(parsed.get("run_id"))
    for k, v in artifact_metrics.items():
        if k not in {"artifact_metric_path", "artifact_payload"} and v is not None:
            parsed[k] = v

    severity, categories = classify_expanded_stderr(stderr, stdout, int(returncode) if returncode is not None else -1)

    expanded_execution_results.append({
        "execution_key": execution_key,
        "strategy_name": row["strategy_name"],
        "window_name": row.get("window_name"),
        "window_start": row.get("window_start"),
        "window_end": row.get("window_end"),
        "command": result.get("command"),
        "enabled": bool(result.get("enabled")),
        "attempted": bool(result.get("enabled")),
        "returncode": returncode,
        "status": result.get("status"),
        "completed": bool(parsed.get("completed")),
        "run_id": parsed.get("run_id") or "",
        "metric_source": metric_source,
        "artifact_metric_path": artifact_metrics.get("artifact_metric_path", ""),
        "cumulative_return": parsed.get("cumulative_return"),
        "benchmark_return": parsed.get("benchmark_return"),
        "excess_return": parsed.get("excess_return"),
        "sharpe_ratio": parsed.get("sharpe_ratio"),
        "trades": parsed.get("trades"),
        "turnover": parsed.get("turnover"),
        "qa_status": parsed.get("qa_status"),
        "qa_rows": parsed.get("qa_rows"),
        "qa_symbols": parsed.get("qa_symbols"),
        "warning_severity": severity,
        "warning_categories": categories,
        "stdout_preview": stdout[:1500],
        "stderr_preview": stderr[:1500],
    })

expanded_execution_results_df = pd.DataFrame(expanded_execution_results)
compact_display(expanded_execution_results_df, rows=100)

if RUN_EXPANDED_STRATEGY_EVALUATION:
    attempted = int(expanded_execution_results_df.get("attempted", pd.Series(dtype=bool)).eq(True).sum()) if not expanded_execution_results_df.empty else 0
    failed = int(expanded_execution_results_df.get("status", pd.Series(dtype=str)).isin(["failed", "exception"]).sum()) if not expanded_execution_results_df.empty else 0
    artifact_metric_rows = int(expanded_execution_results_df.get("metric_source", pd.Series(dtype=str)).isin(["artifact_json", "artifact_csv"]).sum()) if not expanded_execution_results_df.empty else 0
    stdout_metric_rows = int(expanded_execution_results_df.get("metric_source", pd.Series(dtype=str)).eq("stdout").sum()) if not expanded_execution_results_df.empty else 0
    if attempted == 0:
        print("Expanded execution was enabled, but no rows were attempted. Check manual-review gates and candidate selection.")
    elif failed == 0:
        print(f"Expanded execution commands completed successfully for {attempted} attempted row(s).")
        print(f"Expanded metric source rows: artifact={artifact_metric_rows}, stdout={stdout_metric_rows}.")
    else:
        print(f"Expanded execution attempted {attempted} row(s) with {failed} failure(s). Review stderr previews.")


## 16. Load expanded walk-forward artifacts

This cell loads expanded artifacts if available. In review-only or preview-only mode, missing expanded artifacts are expected and recorded as evidence gaps.

In [ ]:
def expanded_metrics_payload_from_execution(row: pd.Series) -> dict[str, Any]:
    payload = {}
    for name in [
        "cumulative_return", "benchmark_return", "excess_return", "sharpe_ratio",
        "trades", "turnover", "qa_rows", "qa_symbols", "correlation",
    ]:
        value = row.get(name)
        if pd.notna(value):
            payload[name] = value
    if row.get("qa_status"):
        payload["qa_status"] = row.get("qa_status")
    if row.get("run_id"):
        payload["run_id"] = row.get("run_id")
    if row.get("metric_source"):
        payload["metric_source"] = row.get("metric_source")
    if row.get("artifact_metric_path"):
        payload["artifact_metric_path"] = row.get("artifact_metric_path")
    return payload


def is_notebook11_interpretive_artifact(path: Path) -> bool:
    """Return True for Notebook 11-generated review-package artifacts.

    These files are useful handoff artifacts, but they must not be counted as
    platform StratLake split/readiness/gate/manifest artifacts.
    """
    try:
        resolved = path.resolve()
        review_dir = NOTEBOOK11_REVIEW_DIR.resolve()
        if review_dir in resolved.parents or resolved == review_dir:
            rel = resolved.relative_to(review_dir)
            parts = set(rel.parts)
            if "expanded_run_review_packages" in parts:
                return True
            if path.name.endswith("_notebook11.json"):
                return True
    except Exception:
        text = path.as_posix()
        if "notebook_11_expanded_promotion_evidence_review/expanded_run_review_packages" in text:
            return True
        if text.endswith("_notebook11.json"):
            return True
    return False


def discover_platform_run_artifact_files(strategy_name: str, run_ids: list[str]) -> list[Path]:
    """Find platform artifacts without self-counting Notebook 11 packages.

    In expanded_run mode, run IDs are stricter than strategy names. A strategy-name
    search can accidentally pick up Notebook 10/restored artifacts for the same
    strategy, so v14 only falls back to strategy-name matching when no run ID is
    available or RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY is explicitly disabled.
    """
    roots = [
        STRATLAKE_ROOT / "artifacts",
        STRATLAKE_ROOT / "reports",
        STRATLAKE_ROOT / "data",
    ]
    run_markers = [str(x) for x in run_ids if x]
    strategy_marker = str(strategy_name) if strategy_name else ""

    if run_markers and RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY:
        artifact_markers = run_markers
    elif run_markers:
        artifact_markers = [t for t in [strategy_marker, *run_markers] if t]
    elif DISCOVER_EXISTING_EXPANDED_PLATFORM_ARTIFACTS:
        # Explicit opt-in for auditing existing artifacts from previous runs.
        artifact_markers = [strategy_marker] if strategy_marker else []
    else:
        # Preview mode should not count stale/restored strategy artifacts as
        # current expanded-run evidence when no run id exists.
        artifact_markers = []

    if not artifact_markers:
        return []

    matches: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            if is_notebook11_interpretive_artifact(p):
                continue
            s = p.as_posix()
            if NOTEBOOK11_REVIEW_DIR.as_posix() in s:
                # Avoid loading Notebook 11 summary/package files as upstream platform artifacts.
                continue
            if any(token in s for token in artifact_markers) and p.suffix.lower() in {".json", ".csv", ".parquet"}:
                matches.append(p)
    return sorted(set(matches), key=lambda p: (len(p.as_posix()), p.as_posix()))


def load_first_platform_json_by_name(files: list[Path], name_patterns: list[str]) -> tuple[dict[str, Any], str]:
    lowered = []
    for p in files:
        if p.suffix.lower() != ".json":
            continue
        if is_notebook11_interpretive_artifact(p):
            continue
        name = p.name.lower()
        if "notebook11" in name or name.endswith("_notebook11.json"):
            continue
        lowered.append((p, name))

    for p, name in lowered:
        if any(pattern in name for pattern in name_patterns):
            payload = safe_load_json(p, default={})
            if payload:
                return payload, p.as_posix()
    return {}, ""


def load_first_platform_split_table(files: list[Path]) -> tuple[pd.DataFrame, str]:
    candidates = [p for p in files if p.suffix.lower() in {".csv", ".parquet"} and not is_notebook11_interpretive_artifact(p)]
    priority = []
    for p in candidates:
        name = p.name.lower()
        score = 0
        if "split" in name:
            score -= 10
        if "metric" in name:
            score -= 5
        priority.append((score, len(p.as_posix()), p))
    for _, _, p in sorted(priority):
        try:
            df = pd.read_parquet(p) if p.suffix.lower() == ".parquet" else pd.read_csv(p)
            cols = {str(c).lower() for c in df.columns}
            if len(df) > 0 and ({"split", "split_id", "window", "fold"} & cols or "split" in p.name.lower()):
                return df, p.as_posix()
        except Exception:
            continue
    return pd.DataFrame(), ""


def build_notebook11_interpretive_package(
    strategy_name: str,
    metrics_json: dict[str, Any],
    item_context: dict[str, Any],
    split_df: pd.DataFrame,
    platform_readiness_json: dict[str, Any],
    platform_promotion_gates_json: dict[str, Any],
    platform_manifest_json: dict[str, Any],
) -> dict[str, Any]:
    """Write notebook-scoped review artifacts without pretending they are platform outputs."""
    run_ids = item_context.get("run_ids", [])
    run_id_label = run_ids[0] if run_ids else "no_run_id"
    package_dir = NOTEBOOK11_REVIEW_DIR / "expanded_run_review_packages" / strategy_name / str(run_id_label)
    package_dir.mkdir(parents=True, exist_ok=True)

    metrics_payload = {
        "source": "notebook11_expanded_run_metric_evidence",
        "strategy_name": strategy_name,
        "run_ids": run_ids,
        "metric_source": item_context.get("metric_source"),
        "metric_paths": item_context.get("metric_paths", []),
        "metrics": metrics_json,
    }

    derived_readiness = {
        "source": "notebook11_interpretive_readiness",
        "strategy_name": strategy_name,
        "status": "needs_more_evidence",
        "metrics_loaded": bool(metrics_json),
        "platform_split_metrics_loaded": isinstance(split_df, pd.DataFrame) and len(split_df) > 0,
        "platform_metrics_readiness_loaded": bool(platform_readiness_json),
        "platform_promotion_gates_loaded": bool(platform_promotion_gates_json),
        "platform_manifest_loaded": bool(platform_manifest_json),
        "reason_codes": [
            reason for reason, active in {
                "metrics_loaded": bool(metrics_json),
                "missing_platform_split_metrics": not (isinstance(split_df, pd.DataFrame) and len(split_df) > 0),
                "missing_platform_metrics_readiness": not bool(platform_readiness_json),
                "missing_platform_promotion_gates": not bool(platform_promotion_gates_json),
                "missing_platform_manifest": not bool(platform_manifest_json),
            }.items() if active
        ],
        "interpretation": "Expanded command metrics are available, but promotion-grade platform review artifacts remain incomplete unless split/readiness/gate artifacts are loaded from StratLake.",
    }

    derived_gate_preview = {
        "source": "notebook11_interpretive_gate_preview",
        "strategy_name": strategy_name,
        "decision": "needs_more_evidence",
        "promotion_grade_claim_made": False,
        "rationale": [
            "Notebook 11 does not replace StratLake promotion gates.",
            "Notebook-scoped gate previews are not promotion decisions.",
            "Split/readiness/gate artifacts should be generated by upstream StratLake surfaces when available.",
        ],
    }

    derived_manifest = {
        "source": "notebook11_interpretive_manifest",
        "strategy_name": strategy_name,
        "created_utc": now_utc(),
        "run_ids": run_ids,
        "package_dir": package_dir.as_posix(),
        "platform_artifacts": {
            "metrics_loaded": bool(metrics_json),
            "split_metric_rows": int(len(split_df)) if isinstance(split_df, pd.DataFrame) else 0,
            "metrics_readiness_loaded": bool(platform_readiness_json),
            "promotion_gates_loaded": bool(platform_promotion_gates_json),
            "manifest_loaded": bool(platform_manifest_json),
        },
        "non_claims": [
            "not_a_core_stratlake_promotion_gate_artifact",
            "not_promotion_grade_evidence",
            "not_strategy_approval",
            "not_statistical_significance",
        ],
    }

    write_json(package_dir / "metrics.json", metrics_payload)
    write_json(package_dir / "metrics_readiness_notebook11.json", derived_readiness)
    write_json(package_dir / "promotion_gate_interpretation_notebook11.json", derived_gate_preview)
    write_json(package_dir / "manifest_notebook11.json", derived_manifest)

    return {
        "notebook11_review_package_created": True,
        "notebook11_review_package_dir": package_dir.as_posix(),
        "notebook11_review_package_status": "complete_interpretive_metrics_package_needs_platform_artifacts",
        "notebook11_metrics_path": (package_dir / "metrics.json").as_posix(),
        "notebook11_readiness_path": (package_dir / "metrics_readiness_notebook11.json").as_posix(),
        "notebook11_gate_preview_path": (package_dir / "promotion_gate_interpretation_notebook11.json").as_posix(),
        "notebook11_manifest_path": (package_dir / "manifest_notebook11.json").as_posix(),
    }


def load_expanded_artifacts_for_strategy(strategy_name: str) -> dict[str, Any]:
    """Load expanded artifacts while separating platform outputs from Notebook 11 packages."""
    exec_rows = (
        expanded_execution_results_df[expanded_execution_results_df["strategy_name"].astype(str) == str(strategy_name)].copy()
        if "expanded_execution_results_df" in globals() and not expanded_execution_results_df.empty
        else pd.DataFrame()
    )

    metrics_from_execution = {}
    run_ids = []
    metric_paths = []
    execution_metric_sources = []
    completed_runs = 0
    attempted_runs = 0

    if not exec_rows.empty:
        attempted_runs = int(exec_rows.get("attempted", pd.Series(dtype=bool)).eq(True).sum())
        completed_runs = int(exec_rows.get("completed", exec_rows.get("status", pd.Series(dtype=str)).eq("ok")).eq(True).sum())
        for _, r in exec_rows.iterrows():
            if r.get("run_id"):
                run_ids.append(str(r.get("run_id")))
            if r.get("artifact_metric_path"):
                metric_paths.append(str(r.get("artifact_metric_path")))
            if r.get("metric_source"):
                execution_metric_sources.append(str(r.get("metric_source")))
            row_payload = expanded_metrics_payload_from_execution(r)
            for k, v in row_payload.items():
                if k not in {"run_id", "metric_source", "artifact_metric_path"} and v is not None and pd.notna(v):
                    metrics_from_execution[k] = v

    current_run_evidence_available = bool(attempted_runs > 0 or completed_runs > 0 or run_ids or metric_paths or metrics_from_execution)

    expected_roots = [
        STRATLAKE_ROOT / "artifacts" / "strategies" / strategy_name,
        STRATLAKE_ROOT / "artifacts" / strategy_name,
        STRATLAKE_ROOT / "artifacts" / "runs" / strategy_name,
    ]
    chosen_root = next((r for r in expected_roots if r.exists()), expected_roots[0])
    expected_artifact_root_exists = bool(chosen_root.exists())

    platform_discovered_files = discover_platform_run_artifact_files(strategy_name, run_ids)
    platform_discovered_files_text = [p.as_posix() for p in platform_discovered_files]

    # Only load expected-root artifacts directly when they exist and are not in a
    # run-id strict expanded-run context. In expanded runs, actual run-linked
    # artifact discovery below is safer than broad strategy-root assumptions.
    load_expected_root_artifacts = (
        bool(expected_artifact_root_exists)
        and (current_run_evidence_available or DISCOVER_EXISTING_EXPANDED_PLATFORM_ARTIFACTS)
        and not (run_ids and RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY)
    )

    metrics_json = safe_load_json(chosen_root / "metrics.json", default={}) if load_expected_root_artifacts else {}
    metrics_by_split = safe_load_csv(chosen_root / "metrics_by_split.csv") if load_expected_root_artifacts else pd.DataFrame()
    metrics_by_split_path = (chosen_root / "metrics_by_split.csv").as_posix() if not metrics_by_split.empty else ""

    platform_metrics_readiness_json = safe_load_json(chosen_root / "metrics_readiness.json", default={}) if load_expected_root_artifacts else {}
    platform_metrics_readiness_path = (chosen_root / "metrics_readiness.json").as_posix() if platform_metrics_readiness_json else ""

    platform_promotion_gates_json = safe_load_json(chosen_root / "promotion_gates.json", default={}) if load_expected_root_artifacts else {}
    platform_promotion_gates_path = (chosen_root / "promotion_gates.json").as_posix() if platform_promotion_gates_json else ""

    platform_manifest_json = safe_load_json(chosen_root / "manifest.json", default={}) if load_expected_root_artifacts else {}
    platform_manifest_path = (chosen_root / "manifest.json").as_posix() if platform_manifest_json else ""

    # Search actual run-linked platform artifact files if expected roots are not the real output location.
    if metrics_by_split.empty:
        metrics_by_split, metrics_by_split_path = load_first_platform_split_table(platform_discovered_files)
    if not platform_metrics_readiness_json:
        platform_metrics_readiness_json, platform_metrics_readiness_path = load_first_platform_json_by_name(platform_discovered_files, ["readiness"])
    if not platform_promotion_gates_json:
        platform_promotion_gates_json, platform_promotion_gates_path = load_first_platform_json_by_name(platform_discovered_files, ["promotion_gate", "promotion-gate", "gate"])
    if not platform_manifest_json:
        platform_manifest_json, platform_manifest_path = load_first_platform_json_by_name(platform_discovered_files, ["manifest", "metadata"])

    # If expected roots are not present, the expanded command can still provide usable metric evidence.
    if not metrics_json and metrics_from_execution:
        metrics_json = metrics_from_execution

    metric_source = "not_available"
    if metric_paths:
        metric_source = "artifact_path"
    elif execution_metric_sources:
        source_set = sorted(set(s for s in execution_metric_sources if s))
        metric_source = ",".join(source_set) if source_set else "not_available"
    elif metrics_json:
        metric_source = "metrics_json"

    platform_complete = (
        bool(metrics_json)
        and int(len(metrics_by_split)) > 0
        and bool(platform_metrics_readiness_json)
        and bool(platform_promotion_gates_json)
    )

    notebook11_package = {
        "notebook11_review_package_created": False,
        "notebook11_review_package_dir": "",
        "notebook11_review_package_status": "not_created",
        "notebook11_metrics_path": "",
        "notebook11_readiness_path": "",
        "notebook11_gate_preview_path": "",
        "notebook11_manifest_path": "",
    }
    if WRITE_NOTEBOOK11_REVIEW_PACKAGES and bool(metrics_json) and current_run_evidence_available:
        notebook11_package = build_notebook11_interpretive_package(
            strategy_name=strategy_name,
            metrics_json=metrics_json,
            item_context={
                "run_ids": run_ids,
                "metric_source": metric_source,
                "metric_paths": metric_paths,
            },
            split_df=metrics_by_split,
            platform_readiness_json=platform_metrics_readiness_json,
            platform_promotion_gates_json=platform_promotion_gates_json,
            platform_manifest_json=platform_manifest_json,
        )

    return {
        "strategy_name": strategy_name,
        "artifact_root": chosen_root.as_posix(),
        "expected_artifact_root_exists": expected_artifact_root_exists,
        "platform_artifact_match_scope": "run_id_strict" if run_ids and RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY else "strategy_or_runid_fallback",
        "load_expected_root_artifacts": bool(load_expected_root_artifacts),
        "platform_discovered_artifact_files": platform_discovered_files_text,
        "run_ids": run_ids,
        "completed_runs": completed_runs,
        "attempted_runs": attempted_runs,
        "metric_paths": metric_paths,
        "current_run_evidence_available": current_run_evidence_available,
        "metric_paths_found": bool(metric_paths),
        "metric_source": metric_source,
        "metrics_json": metrics_json,
        "metrics_by_split": metrics_by_split,
        "metrics_by_split_path": metrics_by_split_path,
        "metrics_readiness_json": platform_metrics_readiness_json,
        "metrics_readiness_path": platform_metrics_readiness_path,
        "promotion_gates_json": platform_promotion_gates_json,
        "promotion_gates_path": platform_promotion_gates_path,
        "manifest_json": platform_manifest_json,
        "manifest_path": platform_manifest_path,
        "platform_metrics_readiness_json": platform_metrics_readiness_json,
        "platform_metrics_readiness_path": platform_metrics_readiness_path,
        "platform_promotion_gates_json": platform_promotion_gates_json,
        "platform_promotion_gates_path": platform_promotion_gates_path,
        "platform_manifest_json": platform_manifest_json,
        "platform_manifest_path": platform_manifest_path,
        "platform_complete_review_artifacts_loaded": platform_complete,
        **notebook11_package,
    }


expanded_artifacts = [load_expanded_artifacts_for_strategy(s) for s in expanded_candidate_names]

expanded_artifact_rows = []
for item in expanded_artifacts:
    split_df = item["metrics_by_split"]
    platform_complete = bool(item.get("platform_complete_review_artifacts_loaded", False))
    notebook_package_created = bool(item.get("notebook11_review_package_created", False))
    if platform_complete:
        completeness = "platform_complete_review_artifacts_loaded"
    elif notebook_package_created:
        completeness = "notebook11_interpretive_package_complete_platform_review_artifacts_incomplete"
    elif bool(item["metrics_json"]):
        completeness = "metrics_loaded_but_platform_review_artifacts_incomplete"
    else:
        completeness = "metrics_missing"

    expanded_artifact_rows.append({
        "strategy_name": item["strategy_name"],
        "artifact_root": item["artifact_root"],
        "expected_artifact_root_exists": item["expected_artifact_root_exists"],
        "platform_artifact_match_scope": item.get("platform_artifact_match_scope", "unknown"),
        "current_run_evidence_available": bool(item.get("current_run_evidence_available", False)),
        "load_expected_root_artifacts": bool(item.get("load_expected_root_artifacts", False)),
        "platform_discovered_artifact_file_count": len(item.get("platform_discovered_artifact_files", [])),
        "platform_discovered_artifact_files_preview": "|".join(item.get("platform_discovered_artifact_files", [])[:5]),
        "run_ids": ",".join(item.get("run_ids", [])),
        "attempted_runs": int(item.get("attempted_runs", 0)),
        "completed_runs": int(item.get("completed_runs", 0)),
        "metric_paths": ",".join(item.get("metric_paths", [])),
        "metric_paths_found": bool(item.get("metric_paths_found", False)),
        "metric_source": item.get("metric_source", "not_available"),
        "metrics_json_loaded": bool(item["metrics_json"]),
        "metrics_by_split_rows": int(len(split_df)) if isinstance(split_df, pd.DataFrame) else 0,
        "metrics_by_split_path": item.get("metrics_by_split_path", ""),
        "metrics_readiness_loaded": bool(item["platform_metrics_readiness_json"]),
        "metrics_readiness_path": item.get("platform_metrics_readiness_path", ""),
        "promotion_gates_loaded": bool(item["platform_promotion_gates_json"]),
        "promotion_gates_path": item.get("platform_promotion_gates_path", ""),
        "manifest_loaded": bool(item["platform_manifest_json"]),
        "manifest_path": item.get("platform_manifest_path", ""),
        "platform_metrics_readiness_loaded": bool(item["platform_metrics_readiness_json"]),
        "platform_metrics_readiness_path": item.get("platform_metrics_readiness_path", ""),
        "platform_promotion_gates_loaded": bool(item["platform_promotion_gates_json"]),
        "platform_promotion_gates_path": item.get("platform_promotion_gates_path", ""),
        "platform_manifest_loaded": bool(item["platform_manifest_json"]),
        "platform_manifest_path": item.get("platform_manifest_path", ""),
        "platform_complete_review_artifacts_loaded": platform_complete,
        "notebook11_review_package_created": notebook_package_created,
        "notebook11_review_package_status": item.get("notebook11_review_package_status", "not_created"),
        "notebook11_review_package_dir": item.get("notebook11_review_package_dir", ""),
        "notebook11_readiness_path": item.get("notebook11_readiness_path", ""),
        "notebook11_gate_preview_path": item.get("notebook11_gate_preview_path", ""),
        "notebook11_manifest_path": item.get("notebook11_manifest_path", ""),
        "artifact_completeness_status": completeness,
    })

expanded_artifact_inventory = pd.DataFrame(expanded_artifact_rows)
compact_display(expanded_artifact_inventory, rows=50)


## 17. Compare Notebook 10 smoke evidence to expanded evidence

The delta table is conservative. Absence of expanded artifacts is labeled `insufficient_expanded_evidence` or `not_evaluated`, not a successful validation.

In [ ]:
def metric_from_payload(payload: dict[str, Any], names: list[str]) -> Any:
    if not isinstance(payload, dict):
        return None
    for name in names:
        if name in payload:
            return payload.get(name)
    # Common nested metric container fallback.
    metrics = payload.get("metrics") if isinstance(payload.get("metrics"), dict) else {}
    for name in names:
        if name in metrics:
            return metrics.get(name)
    return None

expanded_artifact_by_strategy = {item["strategy_name"]: item for item in expanded_artifacts}

delta_rows = []
for _, row in candidate_evidence_table.iterrows():
    strategy = row.get("strategy_name")
    item = expanded_artifact_by_strategy.get(strategy, {})
    metrics = item.get("metrics_json", {}) if isinstance(item, dict) else {}
    split_df = item.get("metrics_by_split", pd.DataFrame()) if isinstance(item, dict) else pd.DataFrame()
    readiness = item.get("metrics_readiness_json", {}) if isinstance(item, dict) else {}
    gates = item.get("promotion_gates_json", {}) if isinstance(item, dict) else {}

    expanded_rows = int(len(split_df)) if isinstance(split_df, pd.DataFrame) else 0
    completed_runs = int(item.get("completed_runs", 0) or 0) if isinstance(item, dict) else 0
    expanded_status = "not_evaluated"
    if item:
        platform_complete = bool(item.get("platform_complete_review_artifacts_loaded", False))
        notebook_package_created = bool(item.get("notebook11_review_package_created", False))
        if platform_complete:
            expanded_status = "platform_complete_review_artifacts_loaded"
        elif notebook_package_created:
            expanded_status = "notebook11_interpretive_package_loaded_platform_artifacts_incomplete"
        elif bool(metrics) or expanded_rows > 0 or bool(readiness) or bool(gates):
            expanded_status = "expanded_metric_artifacts_loaded_review_artifacts_incomplete"
        elif strategy in expanded_candidate_names and completed_runs > 0:
            expanded_status = "expanded_command_succeeded_metrics_unavailable"
        elif strategy in expanded_candidate_names:
            expanded_status = "insufficient_expanded_evidence"

    smoke_decision = row.get("promotion_decision", "unknown")
    gate_status = gates.get("status") or gates.get("decision") or gates.get("result") or "not_available"
    readiness_status = readiness.get("status") or readiness.get("readiness_status") or "not_available"

    interpretation = "not_evaluated"
    if expanded_status == "insufficient_expanded_evidence":
        interpretation = "insufficient_expanded_evidence"
    elif expanded_status == "platform_complete_review_artifacts_loaded":
        if str(gate_status).lower() in {"pass", "eligible", "passed"}:
            interpretation = "eligible_for_human_watchlist_review"
        elif str(gate_status).lower() in {"fail", "blocked", "missing"}:
            interpretation = "blocked_by_expanded_evidence"
        else:
            interpretation = "confirmed_needs_review"
    elif expanded_status in {
        "notebook11_interpretive_package_loaded_platform_artifacts_incomplete",
        "expanded_metric_artifacts_loaded_review_artifacts_incomplete",
    }:
        interpretation = "confirmed_needs_review"

    delta_rows.append({
        "strategy_name": strategy,
        "smoke_decision": smoke_decision,
        "expanded_status": expanded_status,
        "smoke_warnings": row.get("warning_categories", ""),
        "expanded_warnings": "",
        "smoke_metric_source": row.get("metric_source", ""),
        "expanded_metric_source": item.get("metric_source", "artifact_json" if bool(metrics) else "not_available"),
        "smoke_windows": row.get("smoke_window_count", 0),
        "expanded_split_count": expanded_rows,
        "expanded_completed_runs": completed_runs,
        "change_in_cumulative_or_total_return": None,
        "change_in_sharpe_or_selected_metric": None,
        "change_in_qa_or_readiness": readiness_status,
        "promotion_gate_outcome": gate_status,
        "interpretation": interpretation,
    })

smoke_vs_expanded_delta = pd.DataFrame(delta_rows)
compact_display(smoke_vs_expanded_delta, rows=50)

## 18. Promotion evidence sufficiency review

This is Notebook 11's core table. It separates evidence status, gate status, readiness, QA, feature-contract interpretation, artifact completeness, reason codes, and a recommended next action.

In [ ]:
def build_evidence_review_row(row: pd.Series) -> dict[str, Any]:
    strategy = row.get("strategy_name")
    delta_match = smoke_vs_expanded_delta[smoke_vs_expanded_delta["strategy_name"].astype(str) == str(strategy)] if not smoke_vs_expanded_delta.empty else pd.DataFrame()
    delta = delta_match.iloc[0] if not delta_match.empty else pd.Series(dtype=object)

    reason_codes = []
    candidate_label = row.get("candidate_for_expanded_review", "not_evaluated")
    candidate_reason = row.get("expanded_review_reason", "")
    if candidate_reason:
        reason_codes.extend(split_reason_codes(candidate_reason))
    if missing_notebook10_artifacts:
        reason_codes.append("missing_notebook10_artifacts")

    feature_contract_status = "not_evaluated"
    if row.get("missing_requirements") and str(row.get("missing_requirements")).lower() not in {"nan", "none", "", "[]"}:
        feature_contract_status = "blocked_missing_requirements"
    elif candidate_label in {"needs_feature_contract_work", "blocked_runtime_or_missing_features"}:
        feature_contract_status = "needs_feature_contract_work"
    else:
        feature_contract_status = "no_missing_requirements_observed"

    artifact_completeness_status = "not_evaluated"
    if str(delta.get("expanded_status", "")) == "platform_complete_review_artifacts_loaded":
        artifact_completeness_status = "platform_complete_review_artifacts_loaded"
    elif str(delta.get("expanded_status", "")) == "notebook11_interpretive_package_loaded_platform_artifacts_incomplete":
        artifact_completeness_status = "notebook11_interpretive_package_loaded_platform_artifacts_incomplete"
        reason_codes.append("platform_review_artifacts_incomplete")
    elif str(delta.get("expanded_status", "")) == "expanded_metric_artifacts_loaded_review_artifacts_incomplete":
        artifact_completeness_status = "expanded_metric_artifacts_loaded_review_artifacts_incomplete"
        reason_codes.append("platform_review_artifacts_incomplete")
    elif str(delta.get("expanded_status", "")) == "insufficient_expanded_evidence":
        artifact_completeness_status = "expanded_artifacts_missing_or_incomplete"
        reason_codes.append("expanded_artifacts_missing_or_incomplete")
    else:
        artifact_completeness_status = "not_evaluated"

    promotion_gate_status = delta.get("promotion_gate_outcome", "not_available")
    readiness_status = delta.get("change_in_qa_or_readiness", "not_available")
    split_count = int(delta.get("expanded_split_count") or 0) if not pd.isna(delta.get("expanded_split_count", 0)) else 0
    split_stability_status = "not_evaluated" if split_count == 0 else "expanded_splits_available"

    qa_status = "not_evaluated"
    if bool(row.get("qa_is_clean")):
        qa_status = "smoke_qa_clean_observed"
    elif row.get("warning_categories"):
        qa_status = "smoke_warnings_present"
        reason_codes.append("smoke_warnings_present")

    evidence_status = "not_evaluated"
    recommended_next_action = "review_notebook10_context_and_restore_artifacts"

    if candidate_label == "deferred_reference_only":
        evidence_status = "deferred"
        reason_codes.append("reference_only_context_restore_notebook10_artifacts_before_expanded_plan")
        recommended_next_action = "restore_notebook10_artifacts_or_run_expanded_mode_before_candidate_selection"
    elif candidate_label.startswith("blocked") or candidate_label == "needs_feature_contract_work":
        evidence_status = "blocked"
        recommended_next_action = "resolve_blockers_before_expanded_validation"
    elif str(delta.get("interpretation", "")) == "eligible_for_human_watchlist_review":

        evidence_status = "eligible_for_human_watchlist_review"
        recommended_next_action = "human_watchlist_review_not_promotion"
    elif str(delta.get("interpretation", "")) in {"confirmed_needs_review", "improved_but_still_needs_review"}:
        evidence_status = "needs_more_evidence"
        recommended_next_action = "continue_evidence_collection_and_governance_review"
    elif candidate_label == "needs_manual_review":
        evidence_status = "needs_more_evidence"
        recommended_next_action = "manual_review_expanded_outputs_and_continue_evidence_collection"
    elif candidate_label == "expanded_candidate" and NOTEBOOK11_MODE in {"review_only", "expanded_preview"}:
        evidence_status = "deferred"
        recommended_next_action = "run_guarded_expanded_validation_when_ready"
    elif candidate_label == "expanded_candidate":
        evidence_status = "needs_more_evidence"
        recommended_next_action = "inspect_expanded_artifacts_and_governance_outputs"

    return {
        "strategy_name": strategy,
        "evidence_status": evidence_status,
        "promotion_gate_status": promotion_gate_status,
        "readiness_status": readiness_status,
        "split_stability_status": split_stability_status,
        "qa_status": qa_status,
        "feature_contract_status": feature_contract_status,
        "artifact_completeness_status": artifact_completeness_status,
        "reason_codes": "|".join(sorted(set(reason_codes))),
        "recommended_next_action": recommended_next_action,
    }

promotion_evidence_review = pd.DataFrame([build_evidence_review_row(r) for _, r in candidate_evidence_table.iterrows()]) if not candidate_evidence_table.empty else pd.DataFrame()
compact_display(promotion_evidence_review, rows=50)

print("Evidence status counts:")
if not promotion_evidence_review.empty:
    display(promotion_evidence_review["evidence_status"].value_counts(dropna=False).rename_axis("evidence_status").reset_index(name="count"))

## 19. Optional evidence review / governance CLI integration

This section previews or optionally runs existing StratLake governance surfaces. It does not invent a governance schema if no output is produced.

In [ ]:
# Optional evidence-review / governance CLI integration.
#
# Keep execution off by default. This cell first performs optional help/schema
# discovery, then only executes command shapes when the expected arguments are
# actually advertised by the installed StratLake CLI and the specific execution
# flag is enabled.
#
# This avoids the v6 output-cell problem where `stratlake-build-evidence-review
# build --artifact-root ... --output-dir ...` was executed even though the
# installed CLI rejected those arguments.

evidence_review_help_cmd = ["stratlake-build-evidence-review", "build", "--help"]
governance_help_cmd = ["stratlake-run-promotion-governance-report", "--help"]

evidence_review_help_result = run_command(
    evidence_review_help_cmd,
    cwd=STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT,
    enabled=bool(RUN_GOVERNANCE_CLI_SCHEMA_DISCOVERY or RUN_PROMOTION_GOVERNANCE_REPORT),
    timeout=60,
)
governance_help_result = run_command(
    governance_help_cmd,
    cwd=STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT,
    enabled=bool(RUN_GOVERNANCE_CLI_SCHEMA_DISCOVERY or RUN_PROMOTION_GOVERNANCE_REPORT),
    timeout=60,
)

evidence_review_help_text = (
    (evidence_review_help_result.get("stdout") or "") +
    "\n" +
    (evidence_review_help_result.get("stderr") or "")
)
governance_help_text = (
    (governance_help_result.get("stdout") or "") +
    "\n" +
    (governance_help_result.get("stderr") or "")
)

evidence_review_supports_artifact_root = "--artifact-root" in evidence_review_help_text
evidence_review_supports_output_dir = "--output-dir" in evidence_review_help_text

governance_supports_artifact_root = "--artifact-root" in governance_help_text
governance_supports_output_dir = "--output-dir" in governance_help_text
governance_supports_report_id = "--report-id" in governance_help_text
governance_supports_registry_path = "--registry-path" in governance_help_text

evidence_review_cmd = [
    "stratlake-build-evidence-review",
    "build",
    "--artifact-root", (STRATLAKE_ROOT / "artifacts").as_posix(),
    "--output-dir", NOTEBOOK11_REVIEW_DIR.as_posix(),
]

governance_cmd = [
    "stratlake-run-promotion-governance-report",
    "--artifact-root", (STRATLAKE_ROOT / "artifacts").as_posix(),
    "--output-dir", NOTEBOOK11_REVIEW_DIR.as_posix(),
    "--report-id", "notebook_11_expanded_promotion_evidence_review",
]

registry_candidates = [
    STRATLAKE_ROOT / "artifacts" / "experiment_registry.json",
    STRATLAKE_ROOT / "artifacts" / "candidate_registry.json",
    STRATLAKE_ROOT / "artifacts" / "research_campaign_registry.json",
]
governance_registry_path = next((p for p in registry_candidates if p.exists()), None)
if governance_registry_path and governance_supports_registry_path:
    governance_cmd.extend(["--registry-path", governance_registry_path.as_posix()])

evidence_review_execution_supported = (
    evidence_review_supports_artifact_root and evidence_review_supports_output_dir
)
governance_execution_supported = (
    governance_supports_artifact_root and governance_supports_output_dir and governance_supports_report_id
)

evidence_review_enabled = bool(
    RUN_PROMOTION_GOVERNANCE_REPORT and
    RUN_EVIDENCE_REVIEW_CLI_BUILD and
    evidence_review_execution_supported
)
governance_enabled = bool(
    RUN_PROMOTION_GOVERNANCE_REPORT and
    RUN_PROMOTION_GOVERNANCE_REPORT_CLI and
    governance_execution_supported
)

evidence_review_skip_reason = ""
if RUN_PROMOTION_GOVERNANCE_REPORT and RUN_EVIDENCE_REVIEW_CLI_BUILD and not evidence_review_execution_supported:
    evidence_review_skip_reason = "installed_cli_help_does_not_advertise_required_artifact_root_output_dir_arguments"
elif not RUN_EVIDENCE_REVIEW_CLI_BUILD:
    evidence_review_skip_reason = "evidence_review_cli_build_flag_false"

governance_skip_reason = ""
if RUN_PROMOTION_GOVERNANCE_REPORT and RUN_PROMOTION_GOVERNANCE_REPORT_CLI and not governance_execution_supported:
    governance_skip_reason = "installed_cli_help_does_not_advertise_required_artifact_root_output_dir_report_id_arguments"
elif not RUN_PROMOTION_GOVERNANCE_REPORT_CLI:
    governance_skip_reason = "promotion_governance_report_cli_flag_false"

evidence_review_cli_result = run_command(
    evidence_review_cmd,
    cwd=STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT,
    enabled=evidence_review_enabled,
    timeout=60 * 15,
)

governance_cli_result = run_command(
    governance_cmd,
    cwd=STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT,
    enabled=governance_enabled,
    timeout=60 * 15,
)

governance_summary_candidates = [
    NOTEBOOK11_REVIEW_DIR / "promotion_governance_report.json",
    NOTEBOOK11_REVIEW_DIR / "notebook_11_expanded_promotion_evidence_review.json",
    STRATLAKE_ROOT / "artifacts" / "promotion_governance_report.json",
    STRATLAKE_ROOT / "artifacts" / "governance" / "promotion_governance_report.json",
]

governance_summary_path = next((p for p in governance_summary_candidates if p.exists()), None)
governance_summary = safe_load_json(governance_summary_path, default={}) if governance_summary_path else {}

governance_cli_audit = {
    "enabled": bool(RUN_PROMOTION_GOVERNANCE_REPORT),
    "schema_discovery_enabled": bool(RUN_GOVERNANCE_CLI_SCHEMA_DISCOVERY),
    "evidence_review_help_command": preview_command(evidence_review_help_cmd),
    "evidence_review_help_status": evidence_review_help_result.get("status"),
    "evidence_review_help_returncode": evidence_review_help_result.get("returncode"),
    "evidence_review_supports_artifact_root": evidence_review_supports_artifact_root,
    "evidence_review_supports_output_dir": evidence_review_supports_output_dir,
    "evidence_review_execution_supported": evidence_review_execution_supported,
    "evidence_review_execution_enabled": evidence_review_enabled,
    "evidence_review_skip_reason": evidence_review_skip_reason,
    "evidence_review_command": preview_command(evidence_review_cmd),
    "evidence_review_cli_status": evidence_review_cli_result.get("status"),
    "evidence_review_returncode": evidence_review_cli_result.get("returncode"),
    "evidence_review_stderr_preview": (evidence_review_cli_result.get("stderr") or "")[:1000],
    "governance_help_command": preview_command(governance_help_cmd),
    "governance_help_status": governance_help_result.get("status"),
    "governance_help_returncode": governance_help_result.get("returncode"),
    "governance_supports_artifact_root": governance_supports_artifact_root,
    "governance_supports_output_dir": governance_supports_output_dir,
    "governance_supports_report_id": governance_supports_report_id,
    "governance_supports_registry_path": governance_supports_registry_path,
    "governance_execution_supported": governance_execution_supported,
    "governance_execution_enabled": governance_enabled,
    "governance_skip_reason": governance_skip_reason,
    "governance_command": preview_command(governance_cmd),
    "governance_registry_path": governance_registry_path.as_posix() if governance_registry_path else None,
    "governance_cli_status": governance_cli_result.get("status"),
    "governance_returncode": governance_cli_result.get("returncode"),
    "governance_stderr_preview": (governance_cli_result.get("stderr") or "")[:1000],
    "governance_summary_path": governance_summary_path.as_posix() if governance_summary_path else None,
    "governance_summary_loaded": bool(governance_summary),
}

governance_cli_audit


## 20. Caveat and blocker register

The caveat register is intentionally explicit. It records blockers and gaps as evidence sufficiency issues, not notebook failures by default.

In [ ]:
caveat_rows = []

def add_caveat(category: str, severity: str, description: str, affected: str = "notebook", next_action: str = "review"):
    caveat_rows.append({
        "category": category,
        "severity": severity,
        "description": description,
        "affected": affected,
        "next_action": next_action,
    })

for name in missing_notebook10_artifacts:
    add_caveat("missing_notebook10_artifact", "review", f"Notebook 10 artifact missing: {name}", name, "restore_or_rerun_notebook10_artifact_generation")

for caveat in restore_caveats:
    add_caveat("archive_restore", "info", caveat, "archive_restore", "enable_restore_only_when_needed")

if not RUN_EXPANDED_STRATEGY_EVALUATION:
    add_caveat("expanded_execution", "info", "expanded_strategy_execution_not_run_default_off", "expanded_run", "enable_guarded_expanded_run_when_ready")

if not RUN_PROMOTION_GOVERNANCE_REPORT:
    add_caveat("governance", "info", "promotion_governance_cli_not_run_default_off", "governance", "enable_only_after_surface_schema_is_confirmed")


if "notebook10_reference_audit" in globals() and not notebook10_reference_audit.empty:
    for _, r in notebook10_reference_audit.iterrows():
        if str(r.get("status")) not in {"pass"}:
            add_caveat(
                "notebook10_reference_audit",
                "review",
                f"Notebook 10 reference check {r.get('check')} status {r.get('status')}: expected {r.get('expected_notebook10_reference')} observed {r.get('observed_loaded_context')}",
                str(r.get("check")),
                "confirm_restored_notebook10_artifacts_or_update_reference_context",
            )

if expanded_execution_plan.empty:
    add_caveat(
        "expanded_execution_plan",
        "review",
        "expanded_execution_plan_empty",
        "expanded_plan",
        "review_candidate_screening_thresholds_or_restore_notebook10_outputs",
    )

if not candidate_evidence_table.empty:
    for _, r in candidate_evidence_table.iterrows():
        label = r.get("candidate_for_expanded_review")
        if label and label != "expanded_candidate":
            add_caveat(
                "candidate_screening",
                "review" if label == "needs_manual_review" else "blocker",
                f"{r.get('strategy_name')} classified as {label}: {r.get('expanded_review_reason')}",
                str(r.get("strategy_name")),
                "resolve_or_review_before_expanded_validation",
            )

if expanded_artifact_inventory.empty and expanded_candidate_names:
    add_caveat("expanded_artifacts", "review", "expanded_artifact_inventory_empty_for_selected_candidates", "expanded_artifacts", "run_expanded_validation_or_restore_outputs")
elif not expanded_artifact_inventory.empty:
    for _, r in expanded_artifact_inventory.iterrows():
        if not bool(r.get("metrics_readiness_loaded")):
            add_caveat("metrics_readiness", "review", "metrics_readiness_json_missing_or_not_loaded", str(r.get("strategy_name")), "treat_readiness_as_advisory_or_generate_upstream")
        if not bool(r.get("promotion_gates_loaded")):
            add_caveat("promotion_gates", "review", "promotion_gates_json_missing_or_not_loaded", str(r.get("strategy_name")), "confirm_gate_generation_surface")
        if int(r.get("metrics_by_split_rows") or 0) <= 0:
            add_caveat("split_metrics", "review", "metrics_by_split_missing_or_empty", str(r.get("strategy_name")), "generate_or_restore_split_metrics")


if RUN_PROMOTION_GOVERNANCE_REPORT and "governance_cli_audit" in globals():

    if governance_cli_audit.get("evidence_review_execution_supported") is False and governance_cli_audit.get("evidence_review_execution_enabled") is False:
        add_caveat(
            "governance_cli",
            "info",
            "evidence_review_cli_execution_skipped_until_installed_cli_schema_is_confirmed",
            "stratlake-build-evidence-review",
            "review_cli_help_output_before_enabling_evidence_review_build",
        )
    if governance_cli_audit.get("governance_execution_supported") is False and governance_cli_audit.get("governance_execution_enabled") is False:
        add_caveat(
            "governance_cli",
            "info",
            "promotion_governance_cli_execution_skipped_until_installed_cli_schema_is_confirmed",
            "stratlake-run-promotion-governance-report",
            "review_cli_help_output_before_enabling_governance_report",
        )
    if governance_cli_audit.get("evidence_review_cli_status") == "failed":
        add_caveat(
            "governance_cli",
            "review",
            f"evidence_review_cli_failed_returncode_{governance_cli_audit.get('evidence_review_returncode')}",
            "stratlake-build-evidence-review",
            "confirm_current_cli_schema_before_enabling_governance_cell",
        )
    if governance_cli_audit.get("governance_cli_status") == "failed":
        add_caveat(
            "governance_cli",
            "review",
            f"promotion_governance_cli_failed_returncode_{governance_cli_audit.get('governance_returncode')}",
            "stratlake-run-promotion-governance-report",
            "confirm_current_cli_schema_and_required_registry_before_enabling_governance_cell",
        )

add_caveat(
    "promotion_engine_caveat",
    "info",
    "Do not claim fully severity-aware promotion behavior unless confirmed in current StratLake implementation; gates may be binary pass/fail/missing.",
    "promotion_interpretation",
    "keep language as evidence review and human interpretation",
)


if not expanded_artifact_inventory.empty and "notebook11_review_package_created" in expanded_artifact_inventory.columns:
    package_count = int(expanded_artifact_inventory["notebook11_review_package_created"].eq(True).sum())
    platform_complete_count = int(expanded_artifact_inventory.get("platform_complete_review_artifacts_loaded", pd.Series(dtype=bool)).eq(True).sum())
    if package_count > 0 and platform_complete_count < package_count:
        add_caveat(
            "notebook11_interpretive_packages",
            "review",
            "Notebook 11 wrote conservative interpretive review packages for expanded-run metrics, but platform split/readiness/gate artifacts remain incomplete.",
            affected=f"packages={package_count}; platform_complete={platform_complete_count}",
            next_action="treat packages as review aids only; use upstream StratLake surfaces for promotion-grade artifacts",
        )

caveat_register = pd.DataFrame(caveat_rows)
compact_display(caveat_register, rows=100)

## 21. Write Notebook 11 review artifacts

Generated outputs are written only under `artifacts/notebook_11_expanded_promotion_evidence_review/`. Generated runtime artifacts stay out of Git unless the repository explicitly expects fixtures.

In [ ]:
notebook10_smoke_context = {
    "created_utc": now_utc(),
    "notebook10_review_dir": NOTEBOOK10_REVIEW_DIR.as_posix(),
    "observed_context": observed_notebook10_context,
    "missing_notebook10_artifacts": missing_notebook10_artifacts,
    "summary": notebook10_summary,
    "smoke_audit_summary": smoke_audit_summary,
}

write_json(NOTEBOOK11_REVIEW_DIR / "notebook10_smoke_context.json", notebook10_smoke_context)
write_table_pair(notebook10_reference_audit, "notebook10_reference_audit", NOTEBOOK11_REVIEW_DIR)
write_table_pair(candidate_evidence_table, "candidate_evidence_table", NOTEBOOK11_REVIEW_DIR)
write_table_pair(expanded_execution_plan, "expanded_execution_plan", NOTEBOOK11_REVIEW_DIR)
write_table_pair(expanded_artifact_inventory, "expanded_artifact_inventory", NOTEBOOK11_REVIEW_DIR)
write_table_pair(smoke_vs_expanded_delta, "smoke_vs_expanded_delta", NOTEBOOK11_REVIEW_DIR)
write_table_pair(promotion_evidence_review, "promotion_evidence_review", NOTEBOOK11_REVIEW_DIR)
write_table_pair(caveat_register, "caveat_register", NOTEBOOK11_REVIEW_DIR)

expanded_runs_attempted = (
    int(expanded_execution_results_df.get("attempted", expanded_execution_results_df.get("enabled", pd.Series(dtype=bool))).eq(True).sum())
    if not expanded_execution_results_df.empty else 0
)
expanded_runs_completed = (
    int(expanded_execution_results_df.get("status", pd.Series(dtype=str)).eq("ok").sum())
    if not expanded_execution_results_df.empty else 0
)
expanded_runs_failed = (
    int(expanded_execution_results_df.get("status", pd.Series(dtype=str)).isin(["failed", "exception"]).sum())
    if not expanded_execution_results_df.empty else 0
)
manual_review_skipped_count = (
    int(expanded_execution_results_df.get("status", pd.Series(dtype=str)).eq("skipped_manual_review_required").sum())
    if not expanded_execution_results_df.empty else 0
)
preview_only_count = (
    int(expanded_execution_results_df.get("status", pd.Series(dtype=str)).eq("preview_only").sum())
    if not expanded_execution_results_df.empty else 0
)

expanded_artifact_metric_rows = (
    int(expanded_execution_results_df.get("metric_source", pd.Series(dtype=str)).isin(["artifact_json", "artifact_csv", "artifact_path"]).sum())
    if not expanded_execution_results_df.empty else 0
)
expanded_stdout_metric_rows = (
    int(expanded_execution_results_df.get("metric_source", pd.Series(dtype=str)).eq("stdout").sum())
    if not expanded_execution_results_df.empty else 0
)
expanded_metric_rows = expanded_artifact_metric_rows + expanded_stdout_metric_rows

expanded_split_metric_rows = (
    int(expanded_artifact_inventory.get("metrics_by_split_rows", pd.Series(dtype=int)).fillna(0).sum())
    if not expanded_artifact_inventory.empty else 0
)
expanded_metrics_readiness_loaded_count = (
    int(expanded_artifact_inventory.get("platform_metrics_readiness_loaded", expanded_artifact_inventory.get("metrics_readiness_loaded", pd.Series(dtype=bool))).eq(True).sum())
    if not expanded_artifact_inventory.empty else 0
)
expanded_promotion_gates_loaded_count = (
    int(expanded_artifact_inventory.get("platform_promotion_gates_loaded", expanded_artifact_inventory.get("promotion_gates_loaded", pd.Series(dtype=bool))).eq(True).sum())
    if not expanded_artifact_inventory.empty else 0
)
expanded_manifest_loaded_count = (
    int(expanded_artifact_inventory.get("platform_manifest_loaded", expanded_artifact_inventory.get("manifest_loaded", pd.Series(dtype=bool))).eq(True).sum())
    if not expanded_artifact_inventory.empty else 0
)
expanded_complete_review_artifact_count = (
    int(expanded_artifact_inventory.get("platform_complete_review_artifacts_loaded", pd.Series(dtype=bool)).eq(True).sum())
    if not expanded_artifact_inventory.empty else 0
)
notebook11_review_package_count = (
    int(expanded_artifact_inventory.get("notebook11_review_package_created", pd.Series(dtype=bool)).eq(True).sum())
    if not expanded_artifact_inventory.empty else 0
)
notebook11_interpretive_package_incomplete_platform_count = (
    int(expanded_artifact_inventory.get("artifact_completeness_status", pd.Series(dtype=str)).eq("notebook11_interpretive_package_complete_platform_review_artifacts_incomplete").sum())
    if not expanded_artifact_inventory.empty else 0
)


if not RUN_EXPANDED_STRATEGY_EVALUATION:
    expanded_validation_status = "preview_only"
elif expanded_runs_attempted > 0 and expanded_runs_failed == 0 and expanded_complete_review_artifact_count == expanded_runs_completed and expanded_runs_completed > 0:
    expanded_validation_status = "expanded_run_succeeded_complete_review_artifacts"
elif expanded_runs_attempted > 0 and expanded_runs_failed == 0 and notebook11_review_package_count == expanded_runs_completed and expanded_runs_completed > 0:
    expanded_validation_status = "expanded_run_succeeded_with_notebook11_review_packages_platform_artifacts_incomplete"
elif expanded_runs_attempted > 0 and expanded_runs_failed == 0 and expanded_metric_rows > 0:
    expanded_validation_status = "expanded_run_succeeded_with_metrics_review_artifacts_incomplete"
elif expanded_runs_attempted > 0 and expanded_runs_failed == 0:
    expanded_validation_status = "expanded_run_commands_succeeded_metrics_missing"
elif expanded_runs_attempted > 0 and expanded_runs_failed > 0:
    expanded_validation_status = "expanded_run_failed"
elif manual_review_skipped_count > 0:
    expanded_validation_status = "manual_review_required_not_run"
elif expanded_execution_plan.empty:
    expanded_validation_status = "enabled_no_candidates"
else:
    expanded_validation_status = "enabled_no_runs_attempted"

summary_payload = {
    "notebook": "Notebook 11 — StratLake Expanded Promotion Evidence Review",
    "created_utc": now_utc(),
    "notebook11_mode": NOTEBOOK11_MODE,
    "notebook10_context_loaded": bool(notebook10_summary or smoke_audit_summary or not promotion_review.empty),
    "notebook10_context_source": notebook10_context_source,
    "notebook10_runtime_context_loaded": bool(runtime_context_loaded),
    "notebook10_reference_only_context": bool(not runtime_context_loaded and notebook10_context_source == "reference_summary_fallback"),
    "allow_reference_only_expanded_plan": bool(ALLOW_REFERENCE_ONLY_EXPANDED_PLAN),
    "candidate_count": int(len(candidate_evidence_table)),
    "strict_expanded_candidate_count": int(len(strict_expanded_candidate_names)),
    "expanded_candidate_count": int(len(expanded_candidate_names)),
    "expanded_plan_candidate_count": int(len(expanded_candidate_names)),
    "expanded_execution_enabled": bool(RUN_EXPANDED_STRATEGY_EVALUATION),
    "manual_review_candidate_runs_allowed": bool(ALLOW_MANUAL_REVIEW_CANDIDATE_RUNS),
    "run_id_strict_platform_artifact_discovery": bool(RUN_ID_STRICT_PLATFORM_ARTIFACT_DISCOVERY),
    "discover_existing_expanded_platform_artifacts": bool(DISCOVER_EXISTING_EXPANDED_PLATFORM_ARTIFACTS),
    "expanded_runs_attempted": expanded_runs_attempted,
    "expanded_runs_completed": expanded_runs_completed,
    "expanded_runs_failed": expanded_runs_failed,
    "expanded_artifact_metric_rows": expanded_artifact_metric_rows,
    "expanded_stdout_metric_rows": expanded_stdout_metric_rows,
    "expanded_metric_rows": expanded_metric_rows,
    "expanded_split_metric_rows": expanded_split_metric_rows,
    "expanded_platform_metrics_readiness_loaded_count": expanded_metrics_readiness_loaded_count,
    "expanded_metrics_readiness_loaded_count": expanded_metrics_readiness_loaded_count,
    "expanded_platform_promotion_gates_loaded_count": expanded_promotion_gates_loaded_count,
    "expanded_promotion_gates_loaded_count": expanded_promotion_gates_loaded_count,
    "expanded_platform_manifest_loaded_count": expanded_manifest_loaded_count,
    "expanded_manifest_loaded_count": expanded_manifest_loaded_count,
    "expanded_complete_review_artifact_count": expanded_complete_review_artifact_count,
    "notebook11_review_package_count": notebook11_review_package_count,
    "notebook11_interpretive_package_incomplete_platform_count": notebook11_interpretive_package_incomplete_platform_count,
    "platform_review_artifacts_required_for_complete_promotion_evidence": REQUIRE_PLATFORM_REVIEW_ARTIFACTS_FOR_COMPLETE_PROMOTION_EVIDENCE,
    "manual_review_skipped_count": manual_review_skipped_count,
    "preview_only_execution_rows": preview_only_count,
    "promotion_evidence_review_rows": int(len(promotion_evidence_review)),
    "eligible_for_human_watchlist_review_count": int(promotion_evidence_review.get("evidence_status", pd.Series(dtype=str)).eq("eligible_for_human_watchlist_review").sum()) if not promotion_evidence_review.empty else 0,
    "needs_more_evidence_count": int(promotion_evidence_review.get("evidence_status", pd.Series(dtype=str)).eq("needs_more_evidence").sum()) if not promotion_evidence_review.empty else 0,
    "blocked_count": int(promotion_evidence_review.get("evidence_status", pd.Series(dtype=str)).eq("blocked").sum()) if not promotion_evidence_review.empty else 0,
    "deferred_count": int(promotion_evidence_review.get("evidence_status", pd.Series(dtype=str)).eq("deferred").sum()) if not promotion_evidence_review.empty else 0,
    "notebook10_artifact_inventory_rows": int(len(artifact_inventory)) if "artifact_inventory" in globals() and isinstance(artifact_inventory, pd.DataFrame) else 0,
    "expanded_artifact_rows": int(len(expanded_artifact_inventory)),
    "artifact_rows": int(len(expanded_artifact_inventory)),
    "caveat_count": int(len(caveat_register)),
    "promotion_grade_claim_made": False,
    "governance_cli_enabled": bool(RUN_PROMOTION_GOVERNANCE_REPORT),
    "governance_evidence_review_execution_enabled": bool(globals().get("evidence_review_enabled", False)),
    "governance_report_execution_enabled": bool(globals().get("governance_enabled", False)),
    "governance_evidence_review_status": globals().get("governance_cli_audit", {}).get("evidence_review_cli_status") if isinstance(globals().get("governance_cli_audit", {}), dict) else None,
    "governance_report_status": (
        "ok_no_summary_loaded"
        if isinstance(globals().get("governance_cli_audit", {}), dict)
        and globals().get("governance_cli_audit", {}).get("governance_cli_status") == "ok"
        and not globals().get("governance_cli_audit", {}).get("governance_summary_loaded", False)
        else globals().get("governance_cli_audit", {}).get("governance_cli_status")
        if isinstance(globals().get("governance_cli_audit", {}), dict)
        else None
    ),
    "expanded_validation_status": expanded_validation_status,
}

if not summary_payload["notebook10_context_loaded"]:
    summary_payload["handoff_status"] = "insufficient_evidence"
elif summary_payload.get("notebook10_reference_only_context") and NOTEBOOK11_MODE in {"review_only", "expanded_preview"} and not ALLOW_REFERENCE_ONLY_EXPANDED_PLAN:
    summary_payload["handoff_status"] = "expanded_preview_reference_only_context_needs_notebook10_artifacts"
elif NOTEBOOK11_MODE == "review_only":
    summary_payload["handoff_status"] = "review_only_completed"
elif NOTEBOOK11_MODE == "expanded_preview" or not RUN_EXPANDED_STRATEGY_EVALUATION:
    summary_payload["handoff_status"] = "expanded_preview_completed"
elif manual_review_skipped_count > 0 and expanded_runs_attempted == 0:
    summary_payload["handoff_status"] = "expanded_run_skipped_manual_review_required"
elif RUN_EXPANDED_STRATEGY_EVALUATION and expanded_runs_attempted > 0 and expanded_runs_failed == 0 and expanded_complete_review_artifact_count == expanded_runs_completed and expanded_runs_completed > 0:
    summary_payload["handoff_status"] = "expanded_run_completed_with_complete_review_artifacts"
elif RUN_EXPANDED_STRATEGY_EVALUATION and expanded_runs_attempted > 0 and expanded_runs_failed == 0 and expanded_metric_rows > 0:
    summary_payload["handoff_status"] = "expanded_run_completed_with_metrics_review_artifacts_incomplete"
elif RUN_EXPANDED_STRATEGY_EVALUATION and expanded_runs_attempted > 0 and expanded_runs_failed == 0:
    summary_payload["handoff_status"] = "expanded_run_completed_commands_only_needs_artifact_review"
elif RUN_EXPANDED_STRATEGY_EVALUATION and expanded_runs_attempted > 0 and expanded_runs_failed > 0:
    summary_payload["handoff_status"] = "expanded_run_failed_needs_rerun"
else:
    summary_payload["handoff_status"] = "expanded_run_enabled_no_attempts"

write_json(NOTEBOOK11_REVIEW_DIR / "summary.json", summary_payload)
print(json.dumps(summary_payload, indent=2, default=str))


## 22. Optional archive checkpoint

Checkpointing is manual/off by default. Enable only after inspecting Notebook 11 outputs.

In [ ]:
checkpoint_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--archive-id", NOTEBOOK11_ARCHIVE_ID,
    "--archive-collision-policy", "overwrite_allowed",
    "--drive-root", STRATLAKE_DRIVE_ARCHIVE_ROOT.as_posix(),
    "--copy-policy", "overwrite_allowed",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

checkpoint_result = run_command(
    checkpoint_cmd,
    cwd=WORKSPACE_ROOT,
    enabled=RUN_STRATLAKE_ARCHIVE_CHECKPOINT,
    timeout=60 * 30,
)

checkpoint_summary = {
    "enabled": bool(RUN_STRATLAKE_ARCHIVE_CHECKPOINT),
    "status": checkpoint_result.get("status"),
    "returncode": checkpoint_result.get("returncode"),
    "command": checkpoint_result.get("command"),
}
checkpoint_summary

## 23. Final handoff

The final handoff is safe for repository review only after manual runtime review. Runtime handoff values are not committed as proof of execution.

In [ ]:
final_handoff = {
    "notebook": "Notebook 11 — StratLake Expanded Promotion Evidence Review",
    "theme": "From confidence review to promotion evidence.",
    "source_notebook_context": "Notebook 10 — StratLake Walk-Forward Robustness and Promotion Review",
    "source_context_notebook_filename": "Notebook_11_Stratlake_Expanded_Promotion_Evidence_Review_STANDALONE_DRAFT_v17_REFERENCE_CONTEXT_AUDITED.ipynb",
    "later_expected_import_path": "notebooks/11_stratlake_expanded_promotion_evidence_review.ipynb",
    "later_expected_import_milestone": "Milestone 14 — Notebook 11 Expanded Promotion Evidence Review Import",
    "later_expected_import_branch": "features/m14-notebook-11-expanded-promotion-evidence-review",
    "notebook11_mode": summary_payload.get("notebook11_mode"),
    "notebook10_context_loaded": summary_payload.get("notebook10_context_loaded"),
    "notebook10_context_source": summary_payload.get("notebook10_context_source"),
    "notebook10_runtime_context_loaded": summary_payload.get("notebook10_runtime_context_loaded"),
    "notebook10_reference_only_context": summary_payload.get("notebook10_reference_only_context"),
    "allow_reference_only_expanded_plan": summary_payload.get("allow_reference_only_expanded_plan"),
    "candidate_count": summary_payload.get("candidate_count"),
    "strict_expanded_candidate_count": summary_payload.get("strict_expanded_candidate_count"),
    "expanded_candidate_count": summary_payload.get("expanded_candidate_count"),
    "expanded_plan_candidate_count": summary_payload.get("expanded_plan_candidate_count"),
    "notebook10_artifact_inventory_rows": summary_payload.get("notebook10_artifact_inventory_rows"),
    "expanded_execution_enabled": summary_payload.get("expanded_execution_enabled"),
    "run_id_strict_platform_artifact_discovery": summary_payload.get("run_id_strict_platform_artifact_discovery"),
    "discover_existing_expanded_platform_artifacts": summary_payload.get("discover_existing_expanded_platform_artifacts"),
    "expanded_runs_attempted": summary_payload.get("expanded_runs_attempted"),
    "expanded_runs_completed": summary_payload.get("expanded_runs_completed"),
    "expanded_runs_failed": summary_payload.get("expanded_runs_failed"),
    "expanded_artifact_metric_rows": summary_payload.get("expanded_artifact_metric_rows"),
    "expanded_stdout_metric_rows": summary_payload.get("expanded_stdout_metric_rows"),
    "expanded_metric_rows": summary_payload.get("expanded_metric_rows"),
    "expanded_split_metric_rows": summary_payload.get("expanded_split_metric_rows"),
    "expanded_platform_metrics_readiness_loaded_count": summary_payload.get("expanded_platform_metrics_readiness_loaded_count"),
    "expanded_metrics_readiness_loaded_count": summary_payload.get("expanded_metrics_readiness_loaded_count"),
    "expanded_platform_promotion_gates_loaded_count": summary_payload.get("expanded_platform_promotion_gates_loaded_count"),
    "expanded_promotion_gates_loaded_count": summary_payload.get("expanded_promotion_gates_loaded_count"),
    "expanded_platform_manifest_loaded_count": summary_payload.get("expanded_platform_manifest_loaded_count"),
    "expanded_manifest_loaded_count": summary_payload.get("expanded_manifest_loaded_count"),
    "expanded_complete_review_artifact_count": summary_payload.get("expanded_complete_review_artifact_count"),
    "notebook11_review_package_count": summary_payload.get("notebook11_review_package_count"),
    "notebook11_interpretive_package_incomplete_platform_count": summary_payload.get("notebook11_interpretive_package_incomplete_platform_count"),
    "platform_review_artifacts_required_for_complete_promotion_evidence": summary_payload.get("platform_review_artifacts_required_for_complete_promotion_evidence"),
    "manual_review_skipped_count": summary_payload.get("manual_review_skipped_count"),
    "preview_only_execution_rows": summary_payload.get("preview_only_execution_rows"),
    "promotion_evidence_review_rows": summary_payload.get("promotion_evidence_review_rows"),
    "eligible_for_human_watchlist_review_count": summary_payload.get("eligible_for_human_watchlist_review_count"),
    "needs_more_evidence_count": summary_payload.get("needs_more_evidence_count"),
    "blocked_count": summary_payload.get("blocked_count"),
    "deferred_count": summary_payload.get("deferred_count"),
    "artifact_rows": summary_payload.get("artifact_rows"),
    "caveat_count": summary_payload.get("caveat_count"),
    "promotion_grade_claim_made": False,
    "governance_cli_enabled": summary_payload.get("governance_cli_enabled"),
    "governance_evidence_review_execution_enabled": summary_payload.get("governance_evidence_review_execution_enabled"),
    "governance_report_execution_enabled": summary_payload.get("governance_report_execution_enabled"),
    "governance_evidence_review_status": summary_payload.get("governance_evidence_review_status"),
    "governance_report_status": summary_payload.get("governance_report_status"),
    "expanded_validation_status": summary_payload.get("expanded_validation_status"),
    "handoff_status": summary_payload.get("handoff_status"),
    "review_dir": NOTEBOOK11_REVIEW_DIR.as_posix(),
    "non_claims": [
        "no_alpha_claim",
        "no_production_readiness_claim",
        "no_statistical_significance_claim",
        "no_strategy_approval_claim",
        "no_promotion_grade_claim_by_default",
        "no_ci_runtime_equivalence_claim",
    ],
}

write_json(NOTEBOOK11_REVIEW_DIR / "handoff.json", final_handoff)
print(json.dumps(final_handoff, indent=2, default=str))

## 24. Later importation reminder

This repository import is source-only. Keep committed Notebook 11 output-free, execution-count-null, metadata-minimized, and guarded; record manual smoke or expanded evidence only when actually run outside the committed source notebook.